In [ ]:
# PT-BR: # Simulação de Servocontrole Fuzzy Tipo-2 para Sistema MAGLEV
# EN-US: # Type-2 Fuzzy Servocontrol Simulation for a MAGLEV System
# PT-BR: Este notebook implementa e simula o sistema de levitação magnética (MAGLEV) e os controladores fuzzy (Tipo-1 e Tipo-2 Intervalar) conforme descrito na tese de doutorado "Realimentação de Estados baseada em Regras Fuzzy Tipo-2 para Servocontrole de Sistemas Não Lineares" de Missilene da Silva Farias (2019).
# EN-US: This notebook implements and simulates the magnetic levitation (MAGLEV) system and the fuzzy controllers (Type-1 and Interval Type-2) described in Missilene da Silva Farias's 2019 doctoral dissertation, "Type-2 Fuzzy Rule-Based State Feedback for Servocontrol of Nonlinear Systems."

# PT-BR: ## Bloco 1: Importação de Bibliotecas
# EN-US: ## Block 1: Library Imports
# PT-BR: Importamos as bibliotecas necessárias para operações numéricas (NumPy), resolução de equações diferenciais ordinárias (SciPy), projeto de sistemas de controle (SciPy) e plotagem de gráficos (Matplotlib).
# EN-US: The libraries required for numerical operations (NumPy), ordinary differential equation solving (SciPy), control-system design (SciPy), and plotting (Matplotlib) are imported here.
import numpy as np
from scipy.integrate import solve_ivp
# PT-BR: Para alocação de polos
# EN-US: For pole placement
from scipy.signal import place_poles
import matplotlib.pyplot as plt
# PT-BR: Bloco para SALVAR os Resultados Essenciais
# EN-US: Block for SAVING the Essential Results
import pickle
import os
from pathlib import Path
# PT-BR: Caminhos independentes do diretório a partir do qual o notebook foi aberto.
# EN-US: Paths independent of the directory from which the notebook was opened.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = Path.cwd().resolve().parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Execute este notebook dentro do repositório.")
DATA_DIR = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# PT-BR: === BLOCO DE CARREGAMENTO DOS 4 ARQUIVOS PKL (10, 16, 24, 32 bits) ===
# EN-US: === BLOCK FOR LOADING THE 4 PKL FILES (10, 16, 24, 32 bits) ===

bits_list = [10, 16, 24, 32]
files_info = {bits: DATA_DIR / f'maglev_complete_optimization_data_{bits}bits.pkl' for bits in bits_list}

# PT-BR: Dicionário global para armazenar os dados de cada arquivo
# EN-US: Global dictionary that stores the data from each file
all_dados = {}
# PT-BR: Será True se ao menos 1 arquivo carregar
# EN-US: Set to True if at least one file is loaded
DATA_LOADED_SUCCESSFULLY = False

# PT-BR: Armazena quais bits carregaram com sucesso
# EN-US: Stores the bit resolutions loaded successfully
bits_carregados = []

for bits in bits_list:
    results_file_to_load = files_info[bits]
    rotulo = f"QGA {bits} bits"

    print(f"\n{'='*70}")
    print(f"--- Carregando: {rotulo} | Arquivo: {results_file_to_load} ---")
    print(f"{'='*70}")

    if not os.path.exists(results_file_to_load):
        print(f"Arquivo '{results_file_to_load}' nao encontrado. Pulando...")
        all_dados[bits] = None
        continue

    try:
        with open(results_file_to_load, 'rb') as f:
            loaded_data_complete = pickle.load(f)

        # PT-BR: Armazena os dados brutos
        # EN-US: Stores the raw data
        all_dados[bits] = loaded_data_complete

        # PT-BR: Atribui ao escopo global com sufixo de bits para evitar colisões
        # EN-US: Assigns values to the global scope with a bit-resolution suffix to prevent collisions
        # PT-BR: Ex: ga_best_poles_T1_fitness_10bits, ga_best_poles_T1_fitness_16bits, ...
        # EN-US: Example: ga_best_poles_T1_fitness_10bits, ga_best_poles_T1_fitness_16bits, ...
        loaded_keys_list = []
        for key, value in loaded_data_complete.items():
            # PT-BR: variável com sufixo de bits
            # EN-US: Variable with a bit-resolution suffix
            globals()[f"{key}_{bits}bits"] = value
            loaded_keys_list.append(key)

        bits_carregados.append(bits)

        print(f"OK: Carregado com sucesso! Total de variáveis: {len(loaded_keys_list)}")

        # PT-BR: Contagem por categoria
        # EN-US: Count by category
        original_count = len([k for k in loaded_keys_list if k.startswith('original_')])
        ga_count       = len([k for k in loaded_keys_list if k.startswith('ga_')])
        qga_count      = len([k for k in loaded_keys_list if k.startswith('qga_')])
        config_count   = 1 if 'config_info'  in loaded_keys_list else 0
        seed_count     = 1 if 'GLOBAL_SEED'  in loaded_keys_list else 0

        print(f"  Distribuicao: Original={original_count} | GA={ga_count} | QGA={qga_count} | Config+Seed={config_count+seed_count}")

        # PT-BR: Verificações específicas
        # EN-US: Specific checks
        if 'GLOBAL_SEED' in loaded_data_complete:
            print(f"  Seed global: {loaded_data_complete['GLOBAL_SEED']}")

        print(f"\n  Controlador Original:")
        if 'original_local_controllers_K_list' in loaded_data_complete:
            print(f"    K(alpha) originais: {len(loaded_data_complete['original_local_controllers_K_list'])} controladores")
        if 'original_data_log_T1' in loaded_data_complete:
            print(f"    Log T1: {len(loaded_data_complete['original_data_log_T1'].get('t', []))} pontos")
        if 'original_data_log_T2I' in loaded_data_complete:
            print(f"    Log T2I: {len(loaded_data_complete['original_data_log_T2I'].get('t', []))} pontos")

        print(f"\n  GA Standard:")
        if 'ga_best_poles_T1_fitness' in loaded_data_complete:
            print(f"    Melhor ITAE T1 GA: {loaded_data_complete['ga_best_poles_T1_fitness']:.4e}")
        if 'ga_best_poles_T2I_fitness' in loaded_data_complete:
            print(f"    Melhor ITAE T2I GA: {loaded_data_complete['ga_best_poles_T2I_fitness']:.4e}")
        if 'ga_tempo_ga_t1_s' in loaded_data_complete:
            print(f"    Tempo T1 GA: {loaded_data_complete['ga_tempo_ga_t1_s']/60:.1f} min")
        if 'ga_tempo_ga_t2i_s' in loaded_data_complete:
            print(f"    Tempo T2I GA: {loaded_data_complete['ga_tempo_ga_t2i_s']/60:.1f} min")

        print(f"\n  QGA ({bits} bits):")
        if 'qga_melhor_itae_qga_t1' in loaded_data_complete:
            print(f"    Melhor ITAE T1 QGA: {loaded_data_complete['qga_melhor_itae_qga_t1']:.4e}")
        if 'qga_melhor_itae_qga_t2i' in loaded_data_complete:
            print(f"    Melhor ITAE T2I QGA: {loaded_data_complete['qga_melhor_itae_qga_t2i']:.4e}")
        if 'qga_tempo_qga_t1_s' in loaded_data_complete:
            print(f"    Tempo T1 QGA: {loaded_data_complete['qga_tempo_qga_t1_s']/60:.1f} min")
        if 'qga_tempo_qga_t2i_s' in loaded_data_complete:
            print(f"    Tempo T2I QGA: {loaded_data_complete['qga_tempo_qga_t2i_s']/60:.1f} min")

        if 'config_info' in loaded_data_complete:
            cfg = loaded_data_complete['config_info']
            print(f"\n  Config: {cfg.get('timestamp','N/A')} | Seed: {cfg.get('seed_global','N/A')}")
            if 'ga_standard' in cfg:
                print(f"    GA: {cfg['ga_standard'].get('populacao','?')} pop, {cfg['ga_standard'].get('geracoes','?')} gen")
            if 'qga' in cfg:
                print(f"    QGA: {cfg['qga'].get('populacao','?')} pop, {cfg['qga'].get('geracoes','?')} gen")

    except Exception as e:
        print(f"ERRO: Erro ao carregar '{results_file_to_load}': {e}")
        all_dados[bits] = None

# =============================================================================
# PT-BR: Mapeamento de variáveis do arquivo de REFERÊNCIA (primeiro bits carregado)
# EN-US: Variable mapping from the REFERENCE file (first loaded bit resolution)
# PT-BR: para variáveis globais sem sufixo (compatibilidade com demais células)
# EN-US: to global variables without a suffix (compatibility with the remaining cells)
# =============================================================================
if bits_carregados:
    DATA_LOADED_SUCCESSFULLY = True
    # PT-BR: Usa o menor número de bits como referência
    # EN-US: Uses the lowest bit resolution as the reference
    bits_ref = bits_carregados[0]
    loaded_data_ref = all_dados[bits_ref]

    print(f"\n{'='*70}")
    print(f"Mapeando variáveis de referência ({bits_ref} bits) para nomes globais...")
    print(f"{'='*70}")

    mapeamentos = {
        # PT-BR: Controlador Original
        # EN-US: Original Controller
        'original_local_controllers_K_list': 'local_controllers_K_list',
        'original_data_log_T1':              'data_log_T1',
        'original_data_log_T2I':             'data_log_T2I',
        # PT-BR: GA Standard
        # EN-US: Standard GA
        'ga_optimized_desired_poles_T1':     'optimized_desired_poles_T1',
        'ga_optimized_desired_poles_T2I':    'optimized_desired_poles_T2I',
        'ga_best_poles_T1_fitness':          'best_poles_T1_fitness',
        'ga_best_poles_T2I_fitness':         'best_poles_T2I_fitness',
        'ga_data_log_T1_opt':                'data_log_T1_opt',
        'ga_data_log_T2I_opt':               'data_log_T2I_opt',
        'ga_local_controllers_K_list_opt_T1':'local_controllers_K_list_opt_T1',
        'ga_local_controllers_K_list_opt_T2I':'local_controllers_K_list_opt_T2I',
        'ga_historico_ga_t1':                'ga_historico_ga_t1',
        'ga_historico_ga_t2i':               'ga_historico_ga_t2i',
        'ga_tempo_ga_t1_s':                  'ga_tempo_ga_t1_s',
        'ga_tempo_ga_t2i_s':                 'ga_tempo_ga_t2i_s',
        # PT-BR: QGA
        # EN-US: QGA
        'qga_optimized_desired_poles_T1_qga': 'optimized_desired_poles_T1_qga',
        'qga_optimized_desired_poles_T2I_qga':'optimized_desired_poles_T2I_qga',
        'qga_melhor_itae_qga_t1':             'melhor_itae_qga_t1',
        'qga_melhor_itae_qga_t2i':            'melhor_itae_qga_t2i',
        'qga_data_log_T1_opt_qga':            'data_log_T1_opt_qga',
        'qga_data_log_T2I_opt_qga':           'data_log_T2I_opt_qga',
        'qga_local_controllers_K_list_opt_T1_qga': 'local_controllers_K_list_opt_T1_qga',
        'qga_local_controllers_K_list_opt_T2I_qga':'local_controllers_K_list_opt_T2I_qga',
        'qga_historico_qga_t1':               'qga_historico_qga_t1',
        'qga_historico_qga_t2i':              'qga_historico_qga_t2i',
        'qga_tempo_qga_t1_s':                 'qga_tempo_qga_t1_s',
        'qga_tempo_qga_t2i_s':                'qga_tempo_qga_t2i_s',
        'qga_melhores_polos_qga_t1':          'melhores_polos_qga_t1',
        'qga_melhores_polos_qga_t2i':         'melhores_polos_qga_t2i',
    }

    for chave_orig, nome_global in mapeamentos.items():
        if chave_orig in loaded_data_ref:
            globals()[nome_global] = loaded_data_ref[chave_orig]

    # PT-BR: Mapeia também config_info e GLOBAL_SEED
    # EN-US: Also maps config_info and GLOBAL_SEED
    if 'config_info' in loaded_data_ref:
        globals()['config_info'] = loaded_data_ref['config_info']
    if 'GLOBAL_SEED' in loaded_data_ref:
        globals()['GLOBAL_SEED'] = loaded_data_ref['GLOBAL_SEED']

    print(f"  OK: Variáveis mapeadas a partir de {bits_ref} bits.")

# =============================================================================
# PT-BR: Resumo Final
# EN-US: Final Summary
# =============================================================================
print(f"\n{'='*70}")
print("RESUMO DO CARREGAMENTO")
print(f"{'='*70}")
for bits in bits_list:
    status = f"OK: Carregado ({len(all_dados[bits])} chaves)" if all_dados.get(bits) else "ERRO: Falhou/Não encontrado"
    print(f"  {bits:2} bits → {str(files_info[bits]):<52} {status}")

print(f"\nArquivos carregados com sucesso: {bits_carregados}")
print(f"Acesse via: all_dados[10], all_dados[16], all_dados[24], all_dados[32]")
print(f"Variáveis globais mapeadas a partir de: {bits_carregados[0] if bits_carregados else 'nenhum'} bits")

if DATA_LOADED_SUCCESSFULLY:
    print(f"\nOK: Dados carregados! Use DATA_LOADED_SUCCESSFULLY para controlar execuções.")
else:
    print(f"\nERRO: Carregamento falhou para todos os arquivos. Execute as otimizações normalmente.")

In [ ]:
# PT-BR: Definição do Sistema MAGLEV (Capítulo 4.1 da Tese)
# EN-US: MAGLEV System Definition (Dissertation Chapter 4.1)
# PT-BR: Esta seção define a classe `MaglevSystem` que encapsula os parâmetros e a dinâmica do sistema de levitação magnética.
# EN-US: This section defines the `MaglevSystem` class, which encapsulates the parameters and dynamics of the magnetic levitation system.

class MaglevSystem:
    """PT-BR:
    Representa o sistema de levitação magnética (MAGLEV).
    Os parâmetros e equações são baseados no Capítulo 4.1 da tese.

    EN-US:
    Represents the magnetic levitation (MAGLEV) system.
    The parameters and equations are based on dissertation Chapter 4.1.
    """
    def __init__(self, M_b=0.068, R_c=2.8, L_c=0.41, K_m=6.5243e-5, g=9.81, V_max=10.0, x_b_max=0.014, x_b_min_sim=1e-4):
        """PT-BR:
        Inicializa o sistema MAGLEV com seus parâmetros físicos.
        Parâmetros padrão baseados em valores comuns ou da tese (se especificados).
        M_b: massa da esfera (kg)
        R_c: resistência da bobina (Ohm) (Nota: Tese usa Rc+Rs na Eq. 4.1, aqui simplificado ou assumido Rs=0)
        L_c: indutância da bobina (Henry)
        K_m: constante de força do eletroímã (N*m^2/A^2)
        g: aceleração da gravidade (m/s^2)
        V_max: tensão máxima de controle (V)
        x_b_max: posição física máxima da esfera (m) (limite superior do aparato)
        x_b_min_sim: posição mínima para evitar problemas numéricos na simulação (m)

        EN-US:
        Initializes the MAGLEV system with its physical parameters.
        Default parameters are based on commonly used or dissertation values, when specified.
        M_b: sphere mass (kg)
        R_c: coil resistance (ohm) (note: dissertation Eq. 4.1 uses Rc+Rs; here it is simplified or Rs is assumed to be zero)
        L_c: coil inductance (henry)
        K_m: electromagnet force constant (N*m^2/A^2)
        g: gravitational acceleration (m/s^2)
        V_max: maximum control voltage (V)
        x_b_max: maximum physical sphere position (m) (upper apparatus limit)
        x_b_min_sim: minimum position used to avoid numerical problems in the simulation (m)
        """
        self.M_b = M_b
        # PT-BR: Na tese, a Eq. 4.1 usa (Rc + Rs). Aqui consideramos Rs embutido em R_c ou zero.
        # EN-US: Equation 4.1 of the dissertation uses (Rc + Rs). Here, Rs is treated as embedded in R_c or as zero.
        self.R_c = R_c
        self.L_c = L_c
        self.K_m = K_m
        self.g = g
        self.V_max = V_max
        self.x_b_max = x_b_max
        # PT-BR: Para evitar divisão por zero ou instabilidade numérica
        # EN-US: Prevents division by zero or numerical instability
        self.x_b_min_sim = x_b_min_sim
        self.params = {'M_b': M_b, 'R_c': R_c, 'L_c': L_c, 'K_m': K_m, 'g': g}

    def nonlinear_dynamics(self, t, X, Vc_in, r_xb):
        """PT-BR:
        Define as equações diferenciais não lineares do sistema MAGLEV.
        Baseado no modelo não linear do Capítulo 4.1.1 e Eq. 4.5 (com integrador).
        Estados X = [ic, xb, xb_dot, v_int]
        ic: corrente na bobina (A)
        xb: posição da esfera (m)
        xb_dot: velocidade da esfera (m/s)
        v_int: integral do erro de posição (m*s)
        Vc_in: tensão de controle aplicada (V)
        r_xb: referência de posição para a esfera (m)

        EN-US:
        Defines the nonlinear differential equations of the MAGLEV system.
        Based on the nonlinear model in Chapter 4.1.1 and Eq. 4.5 (with integrator).
        States X = [ic, xb, xb_dot, v_int]
        ic: coil current (A)
        xb: sphere position (m)
        xb_dot: sphere velocity (m/s)
        v_int: integral of the position error (m*s)
        Vc_in: applied control voltage (V)
        r_xb: sphere-position reference (m)
        """
        ic, xb, xb_dot, v_int = X
        M_b, R_c, L_c, K_m, g = self.params['M_b'], self.params['R_c'], self.params['L_c'], self.params['K_m'], self.params['g']

        # PT-BR: A tensão de controle Vc_in já deve vir saturada do controlador
        # EN-US: The controller must already have saturated the control voltage Vc_in
        Vc = Vc_in

        # PT-BR: --- Proteções para estabilidade da simulação ---
        # EN-US: --- Simulation-stability safeguards ---
        current_xb = xb
        # PT-BR: Proteção para xb muito baixo ou negativo (instabilidade do modelo não linear)
        # EN-US: Safeguard for a very small or negative xb (nonlinear-model instability)
        if xb <= self.x_b_min_sim:
            # PT-BR: Evita divisão por zero ou valores muito pequenos
            # EN-US: Prevents division by zero or very small values
            current_xb = self.x_b_min_sim
            # PT-BR: Se estiver caindo e no limite inferior
            # EN-US: If the sphere is falling at the lower limit
            if xb_dot < 0:
                # PT-BR: Para a velocidade de queda
                # EN-US: Stops the downward velocity
                xb_dot = 0

        # PT-BR: Equação da corrente (Eq. 4.1 da tese, rearranjada)
        # EN-US: Current equation (dissertation Eq. 4.1, rearranged)
        dic_dt = (Vc - R_c * ic) / L_c
        
        # PT-BR: Definição cinemática
        # EN-US: Kinematic definition
        dxb_dt = xb_dot
        
        # PT-BR: Equação da aceleração da esfera (Eq. 4.3 da tese)
        # EN-US: Sphere-acceleration equation (dissertation Eq. 4.3)
        # PT-BR: Equação alternativa da força magnética, mantida como referência.
        # EN-US: Alternative magnetic-force equation retained for reference.
        # F_mag = (K_m * ic**2) / (2 * current_xb**2) # Força magnética
        # PT-BR: Forma alternativa da aceleração pela segunda lei de Newton, mantida como referência.
        # EN-US: Alternative acceleration form from Newton's second law retained for reference.
        # d(xb_dot)/dt = g - F_mag / M_b # Segunda Lei de Newton
        dxb_dot_dt = g - (K_m * ic**2) / (2 * M_b * current_xb**2)

        # PT-BR: Se a esfera atinge o topo físico (x_b_max), impede que continue subindo
        # EN-US: If the sphere reaches the physical upper bound (x_b_max), prevent further upward motion
        if xb >= self.x_b_max and dxb_dt > 0:
            dxb_dt = 0
            # PT-BR: Se ainda estivesse acelerando para cima
            # EN-US: If it were still accelerating upward
            if dxb_dot_dt > 0:
                dxb_dot_dt = 0
        
        # PT-BR: Se a esfera está no "chão" simulado (x_b_min_sim) e a força resultante a empurra para baixo,
        # EN-US: If the sphere is at the simulated "floor" (x_b_min_sim) and the net force pushes it downward,
        # PT-BR: anula a aceleração e velocidade para baixo para evitar que "afunde" numericamente.
        # EN-US: cancel the downward acceleration and velocity to prevent numerical sinking.
        # PT-BR: O dxb_dt <=0 é redundante se xb_dot já foi zerado acima.
        # EN-US: The dxb_dt <= 0 condition is redundant if xb_dot was already reset above.
        if xb <= self.x_b_min_sim and dxb_dot_dt < 0 and dxb_dt <=0 :
             # PT-BR: Impede que acelere mais para baixo
             # EN-US: Prevents further downward acceleration
             dxb_dot_dt = 0
             # PT-BR: Impede que a posição diminua além do mínimo (já está no mínimo)
             # EN-US: Prevents the position from dropping below the minimum (it is already at the minimum)
             dxb_dt = 0


        # PT-BR: Dinâmica do integrador do erro de posição (para rastreamento de referência - Seção 3.1)
        # EN-US: Position-error integrator dynamics (for reference tracking — Section 3.1)
        dv_int_dt = r_xb - xb

        return [dic_dt, dxb_dt, dxb_dot_dt, dv_int_dt]

    def get_linear_model_matrices(self, xb0, ic0):
        """PT-BR:
        Calcula as matrizes A e B do sistema linearizado aumentado em torno de um ponto de operação (xb0, ic0).
        O modelo linearizado aumentado inclui o integrador do erro.
        Baseado na Eq. 4.8 da Tese (Capítulo 4.1.2).
        Estados: x_aug = [delta_ic, delta_xb, delta_xb_dot, delta_v_int]^T
        Entrada: u = delta_Vc

        EN-US:
        Computes the A and B matrices of the augmented system linearized about an operating point (xb0, ic0).
        The augmented linearized model includes the error integrator.
        Based on Dissertation Eq. 4.8 (Chapter 4.1.2).
        States: x_aug = [delta_ic, delta_xb, delta_xb_dot, delta_v_int]^T
        Input: u = delta_Vc
        """
        M_b, R_c, L_c, K_m, g = self.params['M_b'], self.params['R_c'], self.params['L_c'], self.params['K_m'], self.params['g']

        # PT-BR: Validação do ponto de operação para linearização
        # EN-US: Validates the operating point for linearization
        # PT-BR: ic0 não pode ser zero pela Eq. 4.7 e 4.8
        # EN-US: ic0 cannot be zero according to Eqs. 4.7 and 4.8
        if xb0 <= self.x_b_min_sim or ic0 <= 0:
            print(f"Aviso: Ponto de operação inválido para linearização (xb0={xb0:.4f}, ic0={ic0:.4f}). Retornando matrizes problemáticas.")
            # PT-BR: Retorna matrizes que provavelmente não serão controláveis ou causarão problemas.
            # EN-US: Returns matrices that will probably be uncontrollable or cause problems.
            # PT-BR: Isso idealmente seria tratado antes de chamar esta função.
            # EN-US: Ideally, this condition should be handled before calling this function.
            return np.eye(4) * np.nan, np.zeros((4,1)) * np.nan


        # PT-BR: Matrizes A_lin e B_lin do sistema aumentado [ic, xb, xb_dot, v_int]
        # EN-US: A_lin and B_lin matrices of the augmented system [ic, xb, xb_dot, v_int]
        # PT-BR: Conforme Eq. 4.8 da tese.
        # EN-US: According to dissertation Eq. 4.8.
        # PT-BR: Estados na ordem: x1=ic, x2=xb, x3=xb_dot, x4=v_int (integral do erro de xb)
        # EN-US: State order: x1=ic, x2=xb, x3=xb_dot, x4=v_int (integral of the xb error)
        A_lin = np.array([
            # PT-BR: d(ic)/dt
            # EN-US: d(ic)/dt
            [-R_c/L_c, 0,       0,         0],
            # PT-BR: d(xb)/dt
            # EN-US: d(xb)/dt
            [0,          0,       1,         0],
            # PT-BR: d(xb_dot)/dt (linearizado conforme Eq. 4.7)
            # EN-US: d(xb_dot)/dt (linearized according to Eq. 4.7)
            [-2*g/ic0,   2*g/xb0, 0,         0],
            # PT-BR: d(v_int)/dt = r_xb - xb => d(delta_v_int)/dt = -delta_xb (se r_xb é constante no ponto)
            # EN-US: d(v_int)/dt = r_xb - xb => d(delta_v_int)/dt = -delta_xb (when r_xb is constant at the operating point)
            [0,          -1,      0,         0]
        ])
        B_lin = np.array([
            # PT-BR: Termo de Vc em d(ic)/dt
            # EN-US: Vc term in d(ic)/dt
            [1/L_c],
            [0],
            [0],
            [0]
        ])
        return A_lin, B_lin

# PT-BR: Instanciando o sistema MAGLEV
# EN-US: Instantiates the MAGLEV system
maglev = MaglevSystem()
print("Sistema MAGLEV instanciado.")
print(f"Parâmetros: M_b={maglev.M_b}kg, R_c={maglev.R_c}Ohm, L_c={maglev.L_c}H, K_m={maglev.K_m:.3e}, g={maglev.g}m/s^2")

In [ ]:
# PT-BR: Projeto dos Controladores Locais K(α) (Capítulo 3.1, 4.1.2 da Tese)
# EN-US: Local K(α) Controller Design (Dissertation Chapters 3.1 and 4.1.2)
# PT-BR: Esta seção foca no projeto dos ganhos de realimentação de estados K(α) para diferentes pontos de operação (α) do sistema MAGLEV.
# EN-US: This section focuses on designing the state-feedback gains K(α) for different MAGLEV operating points (α).
# PT-BR: A tese menciona a possibilidade de usar técnicas como posicionamento de polos ou LQR.
# EN-US: The dissertation mentions techniques such as pole placement or LQR.
# PT-BR: Aqui, usaremos o posicionamento de polos (`place_poles`).
# EN-US: Pole placement (`place_poles`) is used here.

# PT-BR: Pontos de operação (Quadro 4.1 da Tese)
# EN-US: Operating points (Dissertation Table 4.1)
# PT-BR: Formato: (xb0 em metros, ic0 em Amperes)
# EN-US: Format: (xb0 in meters, ic0 in amperes)
operating_points = [
    (0.002, 0.286), (0.003, 0.429), (0.004, 0.572), (0.005, 0.715),
    (0.006, 0.858), (0.007, 1.001), (0.008, 1.143), (0.009, 1.286),
    (0.010, 1.429), (0.011, 1.572), (0.012, 1.715)
]
# PT-BR: A tese calcula 11 controladores locais.
# EN-US: The dissertation computes 11 local controllers.

def design_local_controllers(maglev_system_obj, op_points_list):
    """PT-BR:
    Projeta os controladores de realimentação de estados K(alpha) para cada ponto de operação.
    Utiliza alocação de polos para o sistema linearizado aumentado.
    maglev_system_obj: instância da classe MaglevSystem.
    op_points_list: lista de tuplas (xb0, ic0) dos pontos de operação.
    Retorna uma lista de matrizes de ganho K.

    EN-US:
    Designs the K(alpha) state-feedback controllers for each operating point.
    Uses pole placement for the augmented linearized system.
    maglev_system_obj: instance of the MaglevSystem class.
    op_points_list: list of (xb0, ic0) operating-point tuples.
    Returns a list of K gain matrices.
    """
    controllers_K_alpha = []
    
    # PT-BR: Polos desejados para o sistema em malha fechada (exemplo).
    # EN-US: Desired closed-loop poles (example).
    # PT-BR: Estes polos devem ser escolhidos para garantir estabilidade e bom desempenho (resposta rápida, baixo overshoot).
    # EN-US: These poles must be selected to ensure stability and good performance (fast response and low overshoot).
    # PT-BR: Para o sistema aumentado de 4ª ordem [ic, xb, xb_dot, v_int].
    # EN-US: For the fourth-order augmented system [ic, xb, xb_dot, v_int].
    # PT-BR: É crucial que os polos resultem em ganhos K que não saturem excessivamente o atuador.
    # EN-US: The poles must produce K gains that do not excessively saturate the actuator.
    # PT-BR: Polos mais à esquerda no plano complexo geralmente significam resposta mais rápida, mas podem levar a ganhos maiores.
    # EN-US: Poles farther to the left in the complex plane generally yield a faster response but may produce larger gains.
    # PT-BR: Exemplo, pode precisar de ajuste fino
    # EN-US: Example; fine tuning may be required
    desired_poles = np.array([-20, -22, -15+5j, -15-5j])

    print("\nProjetando Controladores Locais K(alpha) via Alocação de Polos:")
    for i, (xb0, ic0) in enumerate(op_points_list):
        print(f"  Ponto de Operação {i+1}: xb0 = {xb0*1000:.1f} mm, ic0 = {ic0:.3f} A")
        A_lin, B_lin = maglev_system_obj.get_linear_model_matrices(xb0, ic0)
        
        if np.isnan(A_lin).any() or np.isnan(B_lin).any():
            print(f"    -> Modelo linear inválido. Usando K nulo.")
            # PT-BR: Ganho nulo
            # EN-US: Zero gain
            K_gain_matrix = np.zeros((1, A_lin.shape[0]))
            controllers_K_alpha.append(K_gain_matrix)
            continue

        try:
            # PT-BR: Verifica controlabilidade do par (A_lin, B_lin)
            # EN-US: Checks controllability of the (A_lin, B_lin) pair
            # PT-BR: A matriz de controlabilidade para um sistema SISO de n estados é C = [B, AB, A^2B, ..., A^(n-1)B]
            # EN-US: For an n-state SISO system, the controllability matrix is C = [B, AB, A^2B, ..., A^(n-1)B]
            n_states = A_lin.shape[0]
            ctrb_matrix_parts = [B_lin]
            for k_i in range(1, n_states):
                ctrb_matrix_parts.append(np.linalg.matrix_power(A_lin, k_i) @ B_lin)
            ctrb_matrix = np.hstack(ctrb_matrix_parts)
            
            rank_ctrb = np.linalg.matrix_rank(ctrb_matrix)
            
            if rank_ctrb < n_states:
                print(f"    -> Sistema NÃO é controlável (posto={rank_ctrb} < {n_states}). Usando K nulo.")
                # PT-BR: Ganho nulo
                # EN-US: Zero gain
                K_gain_matrix = np.zeros((1, n_states))
            else:
                # PT-BR: Projeta o ganho K usando place_poles
                # EN-US: Designs the K gain with place_poles
                # PT-BR: A função place_poles retorna um objeto com o ganho calculado
                # EN-US: The place_poles function returns an object containing the computed gain
                # PT-BR: A lei de controle é u = -K * x_aug
                # EN-US: The control law is u = -K * x_aug
                pp_results = place_poles(A_lin, B_lin, desired_poles)
                # PT-BR: K_gain_matrix é (1 x n_states)
                # EN-US: K_gain_matrix has shape (1 x n_states)
                K_gain_matrix = pp_results.gain_matrix
                print(f"    -> K Projetado = {np.round(K_gain_matrix[0], 3)}")
                # PT-BR: Verifica os autovalores do sistema em malha fechada (A - BK)
                # EN-US: Checks the closed-loop eigenvalues (A - BK)
                # PT-BR: Cálculo opcional dos autovalores de malha fechada, mantido desativado.
                # EN-US: Optional closed-loop eigenvalue calculation retained in disabled form.
                # eigenvalues_closed_loop = np.linalg.eigvals(A_lin - B_lin @ K_gain_matrix)
                # PT-BR: Saída opcional dos autovalores de malha fechada, mantida desativada.
                # EN-US: Optional closed-loop eigenvalue output retained in disabled form.
                # print(f"       Autovalores (A-BK): {np.round(eigenvalues_closed_loop, 2)}")

        except Exception as e:
            print(f"    -> Erro ao projetar K com place_poles: {e}. Usando K nulo.")
            # PT-BR: Ganho nulo
            # EN-US: Zero gain
            K_gain_matrix = np.zeros((1, n_states))
            
        controllers_K_alpha.append(K_gain_matrix)
        
    return controllers_K_alpha

# PT-BR: --- Modificação de `design_local_controllers` ---
# EN-US: --- Modification of `design_local_controllers` ---
# PT-BR: : added desired_poles_input
# EN-US: Record of the addition of desired_poles_input
def design_local_controllers_modifiable_poles(maglev_system_obj, op_points_list, desired_poles_input):
    """PT-BR:
    Projeta os controladores de realimentação de estados K(alpha) para cada ponto de operação.
    Utiliza alocação de polos para o sistema linearizado aumentado.
    maglev_system_obj: instância da classe MaglevSystem.
    op_points_list: lista de tuplas (xb0, ic0) dos pontos de operação.
    desired_poles_input: array numpy com os polos desejados. 
    Retorna uma lista de matrizes de ganho K.

    EN-US:
    Designs the K(alpha) state-feedback controllers for each operating point.
    Uses pole placement for the augmented linearized system.
    maglev_system_obj: instance of the MaglevSystem class.
    op_points_list: list of (xb0, ic0) operating-point tuples.
    desired_poles_input: NumPy array containing the desired poles.
    Returns a list of K gain matrices.
    """
    controllers_K_alpha = []

    print(f"\nProjetando Controladores Locais K(alpha) com Polos: {desired_poles_input}")
    for i, (xb0, ic0) in enumerate(op_points_list):
        # PT-BR: Saída detalhada do ponto de operação, desativada para reduzir a verbosidade durante o GA.
        # EN-US: Detailed operating-point output disabled to reduce verbosity during GA execution.
        #print(f"   Ponto de Operação {i+1}: xb0 = {xb0*1000:.1f} mm, ic0 = {ic0:.3f} A") # Less verbose for GA
        A_lin, B_lin = maglev_system_obj.get_linear_model_matrices(xb0, ic0)

        if np.isnan(A_lin).any() or np.isnan(B_lin).any():
            # PT-BR: Saída de diagnóstico do modelo inválido, desativada para reduzir a verbosidade durante o GA.
            # EN-US: Invalid-model diagnostic output disabled to reduce verbosity during GA execution.
            #print(f"     -> Modelo linear inválido. Usando K nulo.") # Less verbose for GA
            K_gain_matrix = np.zeros((1, A_lin.shape[0]))
            controllers_K_alpha.append(K_gain_matrix)
            # PT-BR: Sinaliza falha para a função de aptidão
            # EN-US: Signals a failure to the fitness function
            return None

        try:
            n_states = A_lin.shape[0]
            ctrb_matrix_parts = [B_lin]
            for k_i in range(1, n_states):
                ctrb_matrix_parts.append(np.linalg.matrix_power(A_lin, k_i) @ B_lin)
            ctrb_matrix = np.hstack(ctrb_matrix_parts)
            rank_ctrb = np.linalg.matrix_rank(ctrb_matrix)

            if rank_ctrb < n_states:
                # PT-BR: Saída de diagnóstico de controlabilidade, desativada para reduzir a verbosidade durante o GA.
                # EN-US: Controllability diagnostic output disabled to reduce verbosity during GA execution.
                #print(f"     -> Sistema NÃO é controlável (posto={rank_ctrb} < {n_states}). Usando K nulo.") # Less verbose for GA
                K_gain_matrix = np.zeros((1, n_states))
                controllers_K_alpha.append(K_gain_matrix)
                # PT-BR: Sinaliza falha para a função de aptidão
                # EN-US: Signals a failure to the fitness function
                return None
            else:
                pp_results = place_poles(A_lin, B_lin, desired_poles_input)
                K_gain_matrix = pp_results.gain_matrix
                # PT-BR: Saída do ganho projetado, desativada para reduzir a verbosidade durante o GA.
                # EN-US: Designed-gain output disabled to reduce verbosity during GA execution.
                # print(f"     -> K Projetado = {np.round(K_gain_matrix[0], 3)}") # Less verbose for GA
        # PT-BR: Captura erros específicos de `place_poles`
        # EN-US: Catches specific place_poles errors
        except ValueError as e:
            # PT-BR: Saída de erro de `place_poles`, desativada para reduzir a verbosidade durante o GA.
            # EN-US: `place_poles` error output disabled to reduce verbosity during GA execution.
            #print(f"     -> Erro ao projetar K com place_poles: {e}. Usando K nulo.") # Less verbose for GA
            K_gain_matrix = np.zeros((1, n_states))
            controllers_K_alpha.append(K_gain_matrix)
            # PT-BR: Sinaliza falha para a função de aptidão
            # EN-US: Signals a failure to the fitness function
            return None
        except Exception as e:
            # PT-BR: Saída de erro inesperado, desativada para reduzir a verbosidade durante o GA.
            # EN-US: Unexpected-error output disabled to reduce verbosity during GA execution.
            #print(f"     -> Erro inesperado ao projetar K: {e}. Usando K nulo.") # Less verbose for GA
            K_gain_matrix = np.zeros((1, n_states))
            controllers_K_alpha.append(K_gain_matrix)
            # PT-BR: Sinaliza falha para a função de aptidão
            # EN-US: Signals a failure to the fitness function
            return None

        controllers_K_alpha.append(K_gain_matrix)

    return controllers_K_alpha


# PT-BR: Projetar os controladores locais
# EN-US: Designs the local controllers
local_controllers_K_list = design_local_controllers(maglev, operating_points)

In [ ]:
# PT-BR: Cálculo e Exibição das Métricas de Desempenho (Capítulo 2.6 da Tese)
# EN-US: Performance-Metric Calculation and Display (Dissertation Chapter 2.6)
# PT-BR: Esta seção calcula e exibe os indicadores de desempenho IAE, ITAE, ITSE e IG para os resultados da simulação.
# EN-US: This section computes and displays the IAE, ITAE, ITSE, and IG performance indicators for the simulation results.
# PT-BR: As definições são baseadas na Seção 2.6 da tese.
# EN-US: The definitions follow dissertation Section 2.6.

# PT-BR: --- Funções para Calcular Métricas de Desempenho ---
# EN-US: --- Functions for Computing Performance Metrics ---
# PT-BR: dt não é mais usado aqui, mas pode manter para consistência da chamada
# EN-US: dt is no longer used here, but it remains in the call signature for consistency
def calculate_iae(error_signal, time_step_dt):
    """PT-BR:
    Calcula a Integral do Módulo do Erro (IAE). Eq. 2.31 (adaptada para discreto).

    EN-US:
    Computes the Integral of Absolute Error (IAE), Eq. 2.31 (adapted to discrete time).
    """
    if len(error_signal) == 0: return np.nan
    N_samples = len(error_signal)
    # PT-BR: Conforme Eq. 2.31 da tese
    # EN-US: According to dissertation Eq. 2.31
    return (1/N_samples) * np.sum(np.abs(error_signal))

# PT-BR: dt não é mais usado aqui
# EN-US: dt is no longer used here
def calculate_itae(error_signal, time_signal_t, time_step_dt):
    """PT-BR:
    Calcula a Integral do Tempo multiplicado pelo Módulo do Erro (ITAE). Eq. 2.32 (adaptada para discreto).

    EN-US:
    Computes the Integral of Time-weighted Absolute Error (ITAE), Eq. 2.32 (adapted to discrete time).
    """
    if len(error_signal) == 0 or len(time_signal_t) == 0: return np.nan
    if len(error_signal) != len(time_signal_t):
        raise ValueError("error_signal e time_signal_t devem ter o mesmo comprimento")
    N_samples = len(error_signal)
    # PT-BR: Conforme Eq. 2.32 da tese
    # EN-US: According to dissertation Eq. 2.32
    return (1/N_samples) * np.sum(time_signal_t * np.abs(error_signal))

# PT-BR: dt não é mais usado aqui
# EN-US: dt is no longer used here
def calculate_itse(error_signal, time_signal_t, time_step_dt):
    """PT-BR:
    Calcula a Integral do Tempo multiplicado pelo Erro Quadrático (ITSE). Eq. 2.33 (adaptada para discreto).

    EN-US:
    Computes the Integral of Time-weighted Squared Error (ITSE), Eq. 2.33 (adapted to discrete time).
    """
    if len(error_signal) == 0 or len(time_signal_t) == 0: return np.nan
    if len(error_signal) != len(time_signal_t):
        raise ValueError("error_signal e time_signal_t devem ter o mesmo comprimento")
    N_samples = len(error_signal)
    # PT-BR: Conforme Eq. 2.33 da tese
    # EN-US: According to dissertation Eq. 2.33
    return (1/N_samples) * np.sum(time_signal_t * (error_signal**2))

# PT-BR: dt não é usado diretamente aqui para os épsilons
# EN-US: dt is not used directly here for the epsilon terms
def calculate_ig(error_signal_s_minus_y, control_signal_u, time_signal_t, alpha1, alpha2, alpha3, time_step_dt):
    """PT-BR:
    Calcula o Índice de Goodhart (IG). Eqs. 2.34-2.37 (adaptadas para discreto).

    EN-US:
    Computes the Goodhart Index (IG), Eqs. 2.34–2.37 (adapted to discrete time).
    """
    if len(error_signal_s_minus_y) == 0 or len(control_signal_u) == 0: return np.nan
    # PT-BR: N_samples usado implicitamente por np.mean e np.var
    # EN-US: N_samples is used implicitly by np.mean and np.var
    N_samples = len(control_signal_u)

    # PT-BR: Eq 2.35: epsilon_1_hat = (1/N) * sum(u(k)) -> Média do sinal de controle.
    # EN-US: Eq. 2.35: epsilon_1_hat = (1/N) * sum(u(k)) -> Mean of the control signal.
    epsilon1_hat_val = np.mean(control_signal_u)
    # PT-BR: Eq 2.36: epsilon_2 = (1/N) * sum((u(k) - epsilon_1_hat)^2) -> Variância do sinal de controle.
    # EN-US: Eq. 2.36: epsilon_2 = (1/N) * sum((u(k) - epsilon_1_hat)^2) -> Variance of the control signal.
    # PT-BR: np.var calcula E[(X - E[X])^2] por padrão, o que corresponde a epsilon2
    # EN-US: By default, np.var computes E[(X - E[X])^2], which corresponds to epsilon2
    epsilon2_val = np.var(control_signal_u)
    # PT-BR: Eq 2.37: epsilon_3 = (1/N) * sum((s(k) - y(k))^2) -> Erro quadrático médio da saída (MSE).
    # EN-US: Eq. 2.37: epsilon_3 = (1/N) * sum((s(k) - y(k))^2) -> Mean squared output error (MSE).
    epsilon3_val = np.mean(error_signal_s_minus_y**2)

    # PT-BR: A tese na Eq. 2.34 usa epsilon1. Se epsilon1 é o mesmo que epsilon1_hat:
    # EN-US: Equation 2.34 of the dissertation uses epsilon1. If epsilon1 is the same as epsilon1_hat:
    # PT-BR: Eq. 2.34
    # EN-US: Eq. 2.34
    ig_index = alpha1 * epsilon1_hat_val + alpha2 * epsilon2_val + alpha3 * epsilon3_val
    return ig_index

In [ ]:
# PT-BR: Simulação (Capítulo 5 da Tese)
# EN-US: Simulation (Dissertation Chapter 5)
# PT-BR: Esta seção executa a simulação do sistema MAGLEV com o controlador fuzzy escolhido (T1 ou T2I).
# EN-US: This section simulates the MAGLEV system with the selected fuzzy controller (T1 or T2I).
# PT-BR: Os resultados são então registrados para plotagem e análise de desempenho.
# EN-US: The results are then logged for plotting and performance analysis.
# PT-BR: A tese compara o desempenho dos SBRFs com controladores de realimentação de estados padrão.
# EN-US: The dissertation compares the performance of the fuzzy rule-based systems with standard state-feedback controllers.

# PT-BR: Variável global para depuração de tempo dentro da ODE (se necessário, mas geralmente não recomendada)
# EN-US: Global variable for debugging time inside the ODE (if required, although this is generally discouraged)
# PT-BR: Variável global de tempo desativada; o tempo é passado explicitamente quando necessário.
# EN-US: Global time variable disabled; time is passed explicitly when needed.
# t_current_sim = 0.0 # Removida pois é melhor passar explicitamente se necessário

# PT-BR: --- Função para Executar Simulação e Coletar Dados ---
# EN-US: --- Function for Running the Simulation and Collecting Data ---
def run_simulation(maglev_obj, controller_obj, X0_sim, t_final_sim, dt_controller_sim, ref_signal_func):
    """PT-BR:
    Executa a simulação do sistema MAGLEV com um dado controlador.
    Retorna um dicionário com os dados logados.

    EN-US:
    Runs a simulation of the MAGLEV system with the given controller.
    Returns a dictionary containing the logged data.
    """
    # PT-BR: Acesso à variável global (se usada para debug na ODE)
    # EN-US: Accesses the global variable (if used for debugging in the ODE)
    global t_current_sim
    
    times_sim = np.arange(0, t_final_sim, dt_controller_sim)
    data_log_sim = {'t': [], 'xb': [], 'r_xb': [], 'Vc': [], 'ic': [], 'xb_dot':[], 'v_int':[]}
    
    current_X_for_sim = X0_sim.copy()
    
    print(f"\nIniciando simulação com controlador: {controller_obj.controller_type}...")
    simulation_successful = True
    for t_idx_sim, t_i_sim in enumerate(times_sim):
        # PT-BR: Atualização opcional do tempo global para depuração, mantida desativada.
        # EN-US: Optional global-time update for debugging retained in disabled form.
        # t_current_sim = t_i_sim # Atualiza tempo global para depuração (se a ODE precisar)
        r_xb_current_val = ref_signal_func(t_i_sim)
        
        # PT-BR: Calcula a ação de controle
        # EN-US: Computes the control action
        Vc_control_signal = controller_obj.calculate_control_action(current_X_for_sim, r_xb_current_val)
        
        # PT-BR: Resolve a dinâmica do sistema para o próximo passo de tempo
        # EN-US: Solves the system dynamics for the next time step
        # PT-BR: Usamos 'args' para passar Vc_control e r_xb_current para maglev.nonlinear_dynamics
        # EN-US: The 'args' parameter passes Vc_control and r_xb_current to maglev.nonlinear_dynamics
        sol_step = solve_ivp(
            maglev_obj.nonlinear_dynamics,
            # PT-BR: Intervalo de integração para este passo
            # EN-US: Integration interval for this time step
            [0, dt_controller_sim],
            # PT-BR: Condição inicial para este passo
            # EN-US: Initial condition for this time step
            current_X_for_sim,
            # PT-BR: Método de integração (Runge-Kutta 4-5)
            # EN-US: Integration method (Runge–Kutta 4–5)
            method='RK45',
            # PT-BR: Argumentos extras para a função de dinâmica
            # EN-US: Additional arguments for the dynamics function
            args=(Vc_control_signal, r_xb_current_val),
            # PT-BR: Não precisamos da solução densa (interpolação)
            # EN-US: Dense output (interpolation) is not required
            dense_output=False,
            # PT-BR: Avalia a solução apenas no final do intervalo
            # EN-US: Evaluates the solution only at the end of the interval
            t_eval=[dt_controller_sim]
        )
        
        # PT-BR: Solução bem-sucedida
        # EN-US: Successful solution
        if sol_step.status == 0:
            # PT-BR: Atualiza o estado para o próximo passo
            # EN-US: Updates the state for the next time step
            current_X_for_sim = sol_step.y[:, -1]
        else:
            print(f"ATENÇÃO: solve_ivp falhou no tempo t={t_i_sim:.3f}s com status {sol_step.status}. Mensagem: {sol_step.message}")
            simulation_successful = False
            # PT-BR: Preenche o restante dos dados com NaN para evitar erros de plotagem de tamanhos diferentes
            # EN-US: Fills the remaining data with NaN to avoid plotting errors caused by arrays of different lengths
            remaining_steps_count = len(times_sim) - t_idx_sim
            nan_fill_array = np.full(remaining_steps_count, np.nan)
            time_remaining_array = times_sim[t_idx_sim:]
            
            data_log_sim['t'].extend(time_remaining_array)
            data_log_sim['ic'].extend(np.full(remaining_steps_count, current_X_for_sim[0] if t_idx_sim > 0 else X0_sim[0]))
            data_log_sim['xb'].extend(np.full(remaining_steps_count, current_X_for_sim[1] if t_idx_sim > 0 else X0_sim[1]))
            data_log_sim['xb_dot'].extend(np.full(remaining_steps_count, current_X_for_sim[2] if t_idx_sim > 0 else X0_sim[2]))
            data_log_sim['v_int'].extend(np.full(remaining_steps_count, current_X_for_sim[3] if t_idx_sim > 0 else X0_sim[3]))
            data_log_sim['Vc'].extend(np.full(remaining_steps_count, Vc_control_signal))
            data_log_sim['r_xb'].extend(ref_signal_func(t) for t in time_remaining_array)
            # PT-BR: Interrompe a simulação
            # EN-US: Stops the simulation
            break

        # PT-BR: Log de dados
        # EN-US: Data log
        data_log_sim['t'].append(t_i_sim)
        data_log_sim['ic'].append(current_X_for_sim[0])
        data_log_sim['xb'].append(current_X_for_sim[1])
        data_log_sim['xb_dot'].append(current_X_for_sim[2])
        data_log_sim['v_int'].append(current_X_for_sim[3])
        # PT-BR: Vc aplicado (já saturado pelo controlador)
        # EN-US: Applied Vc (already saturated by the controller)
        data_log_sim['Vc'].append(Vc_control_signal)
        data_log_sim['r_xb'].append(r_xb_current_val)
        
        # PT-BR: Print de progresso
        # EN-US: Progress output
        # PT-BR: Print a cada ~1 segundo de simulação
        # EN-US: Outputs progress approximately once per simulated second
        if t_idx_sim > 0 and t_idx_sim % (int(1.0/dt_controller_sim)) == 0:
             print(f"  t={t_i_sim:.2f}s, xb={current_X_for_sim[1]*1000:.2f}mm, r_xb={r_xb_current_val*1000:.2f}mm, Vc={Vc_control_signal:.2f}V, ic={current_X_for_sim[0]:.2f}A")

    if simulation_successful:
        print("Simulação concluída com sucesso.")
    else:
        print("Simulação interrompida devido a falha do solver ODE.")
        
    # PT-BR: Converte listas para arrays numpy para facilitar manipulação posterior
    # EN-US: Converts lists to NumPy arrays for easier subsequent manipulation
    for key in data_log_sim:
        data_log_sim[key] = np.array(data_log_sim[key])
        
    return data_log_sim

In [ ]:
# PT-BR: Componentes do Sistema Fuzzy (Capítulo 3.2, 4.1.3, 4.1.4 da Tese)
# EN-US: Fuzzy-System Components (Dissertation Chapters 3.2, 4.1.3, and 4.1.4)
# PT-BR: Esta seção define as funções de pertinência (MFs), a atribuição dos controladores consequentes e a classe do controlador fuzzy.
# EN-US: This section defines the membership functions (MFs), consequent-controller assignment, and fuzzy-controller class.

# PT-BR: --- Funções de Pertinência (MFs) ---
# EN-US: --- Membership Functions (MFs) ---
# PT-BR: Implementação de MFs triangulares e trapezoidais conforme necessidade.
# EN-US: Implements triangular and trapezoidal MFs as required.
# PT-BR: A tese utiliza MFs triangulares e/ou trapezoidais.
# EN-US: The dissertation uses triangular and/or trapezoidal MFs.

def triangular_mf(x, params):
    """PT-BR:
    Calcula o grau de pertinência para uma MF triangular.

    EN-US:
    Computes the membership degree of a triangular MF.
    """
    a, b, c = params
    if not (a <= b <= c): ValueError("Parâmetros da MF triangular inválidos: a <= b <= c deve ser satisfeito.")
    
    val = 0.0
    if a < x < b: val = (x - a) / (b - a)
    elif b < x < c: val = (c - x) / (c - b)
    elif x == b : val = 1.0
    # PT-BR: Casos de borda para MFs "achatadas" onde a=b ou b=c
    # EN-US: Edge cases for "flattened" MFs in which a=b or b=c
    if x == a and a == b: val = 1.0
    if x == c and b == c: val = 1.0
    return np.clip(val, 0.0, 1.0)

def trapezoidal_mf(x, params):
    """PT-BR:
    Calcula o grau de pertinência para uma MF trapezoidal.

    EN-US:
    Computes the membership degree of a trapezoidal MF.
    """
    a, b, c, d = params
    if not (a <= b <= c <= d): ValueError("Parâmetros da MF trapezoidal inválidos: a <= b <= c <= d deve ser satisfeito.")

    val = 0.0
    if a < x < b: val = (x - a) / (b - a)
    elif b <= x <= c: val = 1.0
    elif c < x < d: val = (d - x) / (d - c)
    # PT-BR: Casos de borda
    # EN-US: Edge cases
    if x == a and a == b: val = 1.0
    if x == d and c == d: val = 1.0
    return np.clip(val, 0.0, 1.0)

# PT-BR: Função auxiliar para calcular o valor de uma MF dado seu nome e tipo
# EN-US: Helper function that computes an MF value from its name and type
def calculate_mf_value(x, mf_name, mf_params_dict):
    """PT-BR:
    Calcula o grau de pertinência para uma dada MF.

    EN-US:
    Computes the membership degree for a given MF.
    """
    if mf_name not in mf_params_dict:
        raise ValueError(f"MF com nome '{mf_name}' não encontrada nos parâmetros.")
    
    mf_type, mf_actual_params = mf_params_dict[mf_name]
    
    if mf_type == "triangular":
        return triangular_mf(x, mf_actual_params)
    elif mf_type == "trapezoidal":
        return trapezoidal_mf(x, mf_actual_params)
    else:
        raise ValueError(f"Tipo de MF '{mf_type}' desconhecido para '{mf_name}'.")

# PT-BR: --- Definições das MFs Tipo-1 (T1) para o MAGLEV ---
# EN-US: --- Type-1 (T1) MF Definitions for MAGLEV ---
# PT-BR: Baseado no Quadro 4.2 da Tese.
# EN-US: Based on Dissertation Table 4.2.
# PT-BR: As variáveis fuzzy de entrada para as regras são xb(t) e v_dot_xb(t).
# EN-US: The fuzzy input variables for the rules are xb(t) and v_dot_xb(t).

# PT-BR: MFs para xb(t) - Posição da esfera (valores em metros) - 11 MFs T1
# EN-US: MFs for xb(t) — sphere position (values in meters) — 11 T1 MFs
# PT-BR: Nomes: X2 a X12
# EN-US: Names: X2 through X12
mfs_xb_t1_names = [f"X{i}" for i in range(2, 13)]
# PT-BR: Parâmetros [a,b,c] para triangular, [a,b,c,d] para trapezoidal, convertidos de mm para m.
# EN-US: Parameters [a,b,c] for triangular MFs and [a,b,c,d] for trapezoidal MFs, converted from mm to m.
# PT-BR: Ajustes nos extremos para garantir cobertura e sobreposição adequada.
# EN-US: Endpoint adjustments ensure suitable coverage and overlap.
mfs_xb_t1_params = {
    # PT-BR: Nome: [tipo, [params_em_metros]]
    # EN-US: Name: [type, [parameters_in_meters]]
    # PT-BR: 0-3 mm
    # EN-US: 0–3 mm
    "X2":  ["trapezoidal", [0.0000, 0.0000, 0.0015, 0.0030]],
    # PT-BR: 2-4 mm
    # EN-US: 2–4 mm
    "X3":  ["triangular",  [0.0020, 0.0030, 0.0040]],
    # PT-BR: 3-5 mm
    # EN-US: 3–5 mm
    "X4":  ["triangular",  [0.0030, 0.0040, 0.0050]],
    # PT-BR: 4-6 mm
    # EN-US: 4–6 mm
    "X5":  ["triangular",  [0.0040, 0.0050, 0.0060]],
    # PT-BR: 5-7 mm
    # EN-US: 5–7 mm
    "X6":  ["triangular",  [0.0050, 0.0060, 0.0070]],
    # PT-BR: 6-8 mm
    # EN-US: 6–8 mm
    "X7":  ["triangular",  [0.0060, 0.0070, 0.0080]],
    # PT-BR: 7-9 mm
    # EN-US: 7–9 mm
    "X8":  ["triangular",  [0.0070, 0.0080, 0.0090]],
    # PT-BR: 8-10 mm
    # EN-US: 8–10 mm
    "X9":  ["triangular",  [0.0080, 0.0090, 0.0100]],
    # PT-BR: 9-11 mm
    # EN-US: 9–11 mm
    "X10": ["triangular",  [0.0090, 0.0100, 0.0110]],
    # PT-BR: 10-12 mm
    # EN-US: 10–12 mm
    "X11": ["triangular",  [0.0100, 0.0110, 0.0120]],
    # PT-BR: 11-14 mm
    # EN-US: 11–14 mm
    "X12": ["trapezoidal", [0.0110, 0.0125, 0.0140, 0.0140]],
}

# PT-BR: MFs para v_dot_xb(t) - Erro de posição (valores em metros) - 11 MFs T1
# EN-US: MFs for v_dot_xb(t) — position error (values in meters) — 11 T1 MFs
# PT-BR: O universo de discurso do erro pode ser, por exemplo, de -0.01m a +0.01m.
# EN-US: For example, the error universe of discourse may range from -0.01 m to +0.01 m.
# PT-BR: Ajuste a variável `err_domain_half_width` conforme necessário para cobrir os erros esperados.
# EN-US: Adjust `err_domain_half_width` as needed to cover the expected errors.
# PT-BR: +/- 7mm como exemplo para cobrir um range razoável
# EN-US: +/- 7 mm as an example that covers a reasonable range
err_domain_half_width = 0.007
mfs_v_dot_xb_t1_names = ["Nmax", "Ng", "Nm", "Np", "Nmin", "Z", "Pmin", "Pp", "Pm", "Pg", "Pmax"]
mfs_v_dot_xb_t1_params = {
    # PT-BR: Nome: [tipo, [params_em_metros]]
    # EN-US: Name: [type, [parameters_in_meters]]
    "Nmax": ["trapezoidal", [-2*err_domain_half_width, -2*err_domain_half_width, -1.5*err_domain_half_width, -1.0*err_domain_half_width]],
    "Ng":   ["triangular",  [-1.6*err_domain_half_width, -1.0*err_domain_half_width, -0.7*err_domain_half_width]],
    "Nm":   ["triangular",  [-1.0*err_domain_half_width, -0.7*err_domain_half_width, -0.4*err_domain_half_width]],
    # PT-BR: Pequeno Negativo na Tese é Np
    # EN-US: The Small Negative set in the dissertation is Np
    "Np":   ["triangular",  [-0.7*err_domain_half_width, -0.4*err_domain_half_width, -0.1*err_domain_half_width]],
    "Nmin": ["triangular",  [-0.4*err_domain_half_width, -0.1*err_domain_half_width,  0.0*err_domain_half_width]],
    # PT-BR: Zero (estreito)
    # EN-US: Zero (narrow)
    "Z":    ["triangular",  [-0.15*err_domain_half_width, 0.0*err_domain_half_width,  0.15*err_domain_half_width]],
    "Pmin": ["triangular",  [ 0.0*err_domain_half_width,  0.1*err_domain_half_width,  0.4*err_domain_half_width]],
    "Pp":   ["triangular",  [ 0.1*err_domain_half_width,  0.4*err_domain_half_width,  0.7*err_domain_half_width]],
    "Pm":   ["triangular",  [ 0.4*err_domain_half_width,  0.7*err_domain_half_width,  1.0*err_domain_half_width]],
    "Pg":   ["triangular",  [ 0.7*err_domain_half_width,  1.0*err_domain_half_width,  1.6*err_domain_half_width]],
    "Pmax": ["trapezoidal", [ 1.0*err_domain_half_width,  1.5*err_domain_half_width,  2*err_domain_half_width,  2*err_domain_half_width]],
}
print(f"Definidas {len(mfs_xb_t1_names)} MFs para xb(t) e {len(mfs_v_dot_xb_t1_names)} MFs para v_dot_xb(t) para o controlador T1.")

# PT-BR: --- Atribuição dos Controladores Consequentes K(α) às Regras ---
# EN-US: --- Assignment of Consequent K(α) Controllers to the Rules ---
# PT-BR: Baseado na Seção 3.2.1 (Sistema de Inferência Fuzzy Proposto) e Eq. 3.9.
# EN-US: Based on Section 3.2.1 (Proposed Fuzzy Inference System) and Eq. 3.9.
# PT-BR: Para cada regra "SE posição é MF_xb E erro é MF_v_dot ENTÃO K_consequente",
# EN-US: For each rule "IF position is MF_xb AND error is MF_v_dot THEN K_consequent,"
# PT-BR: K_consequente é um dos K(α) projetados.
# EN-US: K_consequent is one of the designed K(α) gains.
# PT-BR: A escolha de K(α) para a regra é baseada na estimativa da referência r(t) = xb(t) + v_dot_xb(t)
# EN-US: The choice of K(α) for a rule is based on the reference estimate r(t) = xb(t) + v_dot_xb(t)
# PT-BR: que seria ativada por aquela combinação de MFs antecedentes.
# EN-US: that would be activated by that combination of antecedent MFs.
def assign_consequent_controllers(maglev_sys,
                                  current_mf_names_xb, current_mf_params_xb,
                                  current_mf_names_v_dot, current_mf_params_v_dot,
                                  local_K_alpha_gains_list, op_points_for_K_list):
    """PT-BR:
    Atribui um controlador K(alpha) local ao consequente de cada regra fuzzy.
    A tese propõe estimar a referência r(t) para a regra e escolher o K(alpha) do ponto de operação mais próximo.

    EN-US:
    Assigns a local K(alpha) controller to the consequent of each fuzzy rule.
    The dissertation proposes estimating the rule reference r(t) and selecting the K(alpha)
    associated with the nearest operating point.
    """
    rule_consequent_K_list = []
    # PT-BR: Lista apenas dos valores xb0 dos K(alpha)
    # EN-US: List containing only the xb0 values of the K(alpha) gains
    op_points_xb0_values = [op[0] for op in op_points_for_K_list]

    print(f"\n  Atribuindo K(alpha) para {len(current_mf_names_xb) * len(current_mf_names_v_dot)} regras fuzzy:")
    
    rule_counter = 0
    for mf_xb_name_iter in current_mf_names_xb:
        # PT-BR: Obtém os limites da MF para xb (min_xb_mf, max_xb_mf)
        # EN-US: Obtains the xb MF bounds (min_xb_mf, max_xb_mf)
        # PT-BR: Para triangular [a,b,c], usamos [a,c]. Para trapezoidal [a,b,c,d], usamos [a,d].
        # EN-US: For triangular [a,b,c], [a,c] is used; for trapezoidal [a,b,c,d], [a,d] is used.
        params_xb_list_current = current_mf_params_xb[mf_xb_name_iter][1]
        min_xb_mf_val = params_xb_list_current[0]
        max_xb_mf_val = params_xb_list_current[-1]

        for mf_v_dot_name_iter in current_mf_names_v_dot:
            params_v_dot_list_current = current_mf_params_v_dot[mf_v_dot_name_iter][1]
            min_v_dot_mf_val = params_v_dot_list_current[0]
            max_v_dot_mf_val = params_v_dot_list_current[-1]
            
            # PT-BR: Estimativa da referência r(t) usando a matriz mL^j (Eq. 3.9 da tese)
            # EN-US: Estimates the reference r(t) with the mL^j matrix (dissertation Eq. 3.9)
            # PT-BR: mL^j = [[min(y)+min(v_dot), min(y)+max(v_dot)],
            # EN-US: mL^j = [[min(y)+min(v_dot), min(y)+max(v_dot)],
            # PT-BR: [max(y)+min(v_dot), max(y)+max(v_dot)]]
            # EN-US: [max(y)+min(v_dot), max(y)+max(v_dot)]]
            # PT-BR: Onde y é xb(t) aqui.
            # EN-US: Here, y is xb(t).
            # PT-BR: A média dos elementos de mL^j (med(mL^j)) é usada para estimar r(t).
            # EN-US: The mean of the mL^j elements (med(mL^j)) estimates r(t).
            ref_estimates = [
                min_xb_mf_val + min_v_dot_mf_val, min_xb_mf_val + max_v_dot_mf_val,
                max_xb_mf_val + min_v_dot_mf_val, max_xb_mf_val + max_v_dot_mf_val
            ]
            mean_ref_estimate_for_rule = np.mean(ref_estimates)
            
            # PT-BR: Limita a referência estimada ao intervalo dos xb0 para os quais temos K(alpha)
            # EN-US: Clips the estimated reference to the xb0 interval for which K(alpha) gains are available
            # PT-BR: Isso garante que sempre encontraremos um K(alpha) "próximo".
            # EN-US: This ensures that a nearby K(alpha) gain is always found.
            mean_ref_estimate_clipped = np.clip(mean_ref_estimate_for_rule,
                                                min(op_points_xb0_values),
                                                max(op_points_xb0_values))
            
            # PT-BR: Encontra o índice do K(alpha) cujo xb0 é o mais próximo da referência estimada e clipada
            # EN-US: Finds the K(alpha) index whose xb0 is closest to the clipped reference estimate
            closest_K_op_idx = np.argmin(np.abs(np.array(op_points_xb0_values) - mean_ref_estimate_clipped))
            selected_K_for_rule = local_K_alpha_gains_list[closest_K_op_idx]
            rule_consequent_K_list.append(selected_K_for_rule)
            rule_counter += 1
            
            # PT-BR: Descomente para depuração detalhada da atribuição de K
            # EN-US: Uncomment for detailed debugging of the K-gain assignment
            # PT-BR: Saída opcional para depurar a associação entre regra e controlador consequente.
            # EN-US: Optional output for debugging the association between each rule and its consequent controller.
            # print(f"    Regra ({mf_xb_name_iter}, {mf_v_dot_name_iter}): r_est={mean_ref_estimate_for_rule*1000:.1f}mm -> K do OP xb0={op_points_xb0_values[closest_K_op_idx]*1000:.1f}mm")

    print(f"    Total de {rule_counter} controladores de consequente atribuídos para as regras.")
    return rule_consequent_K_list


# PT-BR: --- Classe do Controlador Fuzzy ---
# EN-US: --- Fuzzy Controller Class ---
class FuzzyController:
    """PT-BR:
    Implementa o controlador fuzzy (T1 ou T2I) para o sistema MAGLEV.
    A estrutura segue o SBRF proposto na Seção 3.2 da tese.

    EN-US:
    Implements the fuzzy controller (T1 or T2I) for the MAGLEV system.
    The structure follows the fuzzy rule-based system proposed in dissertation Section 3.2.
    """
    def __init__(self, maglev_system_obj, local_K_gains, op_points_definitions,
                 # PT-BR: Pode ser dict (T1) ou dict de dicts {'umf':..., 'lmf':...} (T2I)
                 # EN-US: May be a dictionary (T1) or a dictionary of dictionaries {'umf':..., 'lmf':...} (T2I)
                 mf_names_for_xb, mf_params_for_xb,
                 # PT-BR: Idem
                 # EN-US: Same as above
                 mf_names_for_v_dot, mf_params_for_v_dot,
                 controller_type_str="T1"):
        self.maglev = maglev_system_obj
        # PT-BR: Lista de todos os K(alpha) projetados
        # EN-US: List of all designed K(alpha) gains
        self.local_K_alphas_all = local_K_gains
        # PT-BR: Lista de tuplas (xb0, ic0)
        # EN-US: List of (xb0, ic0) tuples
        self.operating_points_all = op_points_definitions
        
        self.mfs_xb_names_list = mf_names_for_xb
        self.mfs_v_dot_xb_names_list = mf_names_for_v_dot
        self.controller_type = controller_type_str.upper()

        if self.controller_type == "T1":
            self.current_mfs_xb_parameters = mf_params_for_xb
            self.current_mfs_v_dot_xb_parameters = mf_params_for_v_dot
            self.rule_consequents_K_values = assign_consequent_controllers(
                self.maglev, self.mfs_xb_names_list, self.current_mfs_xb_parameters,
                self.mfs_v_dot_xb_names_list, self.current_mfs_v_dot_xb_parameters,
                self.local_K_alphas_all, self.operating_points_all
            )
        elif self.controller_type == "T2I":
            # PT-BR: Para T2I, precisamos de UMFs e LMFs
            # EN-US: UMFs and LMFs are required for T2I
            self.mfs_xb_params_umf_set = mf_params_for_xb['umf']
            self.mfs_xb_params_lmf_set = mf_params_for_xb['lmf']
            self.mfs_v_dot_xb_params_umf_set = mf_params_for_v_dot['umf']
            self.mfs_v_dot_xb_params_lmf_set = mf_params_for_v_dot['lmf']
            # PT-BR: A tese sugere que a atribuição de K(alpha) pode usar MFs T1 ou as UMFs do T2I.
            # EN-US: The dissertation suggests that K(alpha) assignment may use either T1 MFs or the T2I UMFs.
            # PT-BR: Aqui, usaremos as UMFs para a atribuição, conforme o espírito de usar a fronteira da FOU.
            # EN-US: The UMFs are used here, consistent with using the FOU boundary.
            print("  Atribuindo K(alpha) para T2I (baseado em UMFs):")
            self.rule_consequents_K_values = assign_consequent_controllers(
                self.maglev, self.mfs_xb_names_list, self.mfs_xb_params_umf_set,
                self.mfs_v_dot_xb_names_list, self.mfs_v_dot_xb_params_umf_set,
                self.local_K_alphas_all, self.operating_points_all
            )
        else:
            raise ValueError(f"Tipo de controlador '{self.controller_type}' desconhecido.")
        
        print(f"Controlador Fuzzy {self.controller_type} inicializado com {len(self.rule_consequents_K_values)} regras.")

    def _calculate_single_path_output(self, xb_input_val, v_dot_xb_input_val,
                                      # PT-BR: Vetor de estados [ic, xb, xb_dot, v_int]
                                      # EN-US: State vector [ic, xb, xb_dot, v_int]
                                      current_state_vector_phi,
                                      mf_params_xb_set, mf_params_v_dot_set):
        """PT-BR:
        Calcula a saída de um único caminho do sistema de inferência fuzzy (para T1, ou para UMF/LMF do T2I).
        current_state_vector_phi: é o vetor φ(t) = [x(t), v(t), v_dot(t), y(t), c] da Eq. 3.2.
                                  No MAGLEV, y(t)=xb(t). v_dot(t) não está no vetor de estados,
                                  mas é usado para selecionar MFs.
                                  Para o consequente Z_j = K(α)^j φ(t), o φ(t) usado no produto por K
                                  é o vetor de estados aumentado [ic, xb, xb_dot, v_int].

        EN-US:
        Computes the output of one fuzzy-inference path (for T1 or a T2I UMF/LMF path).
        current_state_vector_phi: vector φ(t) = [x(t), v(t), v_dot(t), y(t), c] from Eq. 3.2.
                                  For MAGLEV, y(t)=xb(t). v_dot(t) is not in the state vector,
                                  but it is used to select the MFs.
                                  For the consequent Z_j = K(α)^j φ(t), the φ(t) multiplied by K
                                  is the augmented state vector [ic, xb, xb_dot, v_int].
        """
        firing_strengths_Rj_list = []
        rule_individual_outputs_Zj_list = []
        
        rule_iterator_idx = 0
        for mf_xb_name_val in self.mfs_xb_names_list:
            mu_xb_val = calculate_mf_value(xb_input_val, mf_xb_name_val, mf_params_xb_set)
            for mf_v_dot_name_val in self.mfs_v_dot_xb_names_list:
                mu_v_dot_xb_val = calculate_mf_value(v_dot_xb_input_val, mf_v_dot_name_val, mf_params_v_dot_set)
                
                # PT-BR: t-norma (mínimo) para o grau de ativação da regra R_j (Definição 3.3)
                # EN-US: t-norm (minimum) for the activation degree of rule R_j (Definition 3.3)
                R_j_activation = min(mu_xb_val, mu_v_dot_xb_val)
                firing_strengths_Rj_list.append(R_j_activation)
                
                # PT-BR: Consequente da regra: Z_j(t) = K(α)^j * φ(t) (Eq. 3.11)
                # EN-US: Rule consequent: Z_j(t) = K(α)^j * φ(t) (Eq. 3.11)
                # PT-BR: Onde K(α)^j é o ganho local atribuído a esta regra, e φ(t) é o vetor de estados aumentado.
                # EN-US: K(α)^j is the local gain assigned to this rule, and φ(t) is the augmented state vector.
                # PT-BR: A tese (Eq. 3.12) detalha K(α)φ(t) = -Kx + K_I v + K_e v_dot + K_y y + K_c c.
                # EN-US: Dissertation Eq. 3.12 expands K(α)φ(t) = -Kx + K_I v + K_e v_dot + K_y y + K_c c.
                # PT-BR: Se K_j_pp já é o ganho para u = -K_pp * X_aug, então Z_j = -K_pp * X_aug.
                # EN-US: If K_j_pp is already the gain for u = -K_pp * X_aug, then Z_j = -K_pp * X_aug.
                # PT-BR: K_j_for_rule é (1 x n_states)
                # EN-US: K_j_for_rule has shape (1 x n_states)
                K_j_for_rule = self.rule_consequents_K_values[rule_iterator_idx]
                
                # PT-BR: current_state_vector_phi = [ic_abs, xb_abs, xb_dot_abs, v_int_abs]
                # EN-US: current_state_vector_phi = [ic_abs, xb_abs, xb_dot_abs, v_int_abs]
                # PT-BR: Produto escalar: K_j[0]*st[0] + K_j[1]*st[1] + ...
                # EN-US: Dot product: K_j[0]*st[0] + K_j[1]*st[1] + ...
                # PT-BR: Z_j é a AÇÃO DE CONTROLE (Vc) proposta por esta regra.
                # EN-US: Z_j is the CONTROL ACTION (Vc) proposed by this rule.
                Z_j_output = -np.dot(K_j_for_rule[0], current_state_vector_phi)
                
                rule_individual_outputs_Zj_list.append(Z_j_output)
                rule_iterator_idx += 1
        
        # PT-BR: Agregação e Defuzzificação (para Takagi-Sugeno de primeira ordem ou tipo zero, é uma média ponderada)
        # EN-US: Aggregation and defuzzification (a weighted mean for first-order or zero-order Takagi–Sugeno)
        # PT-BR: u_F(t) = sum(R_j * Z_j) / sum(R_j) (Eq. 3.13)
        # EN-US: u_F(t) = sum(R_j * Z_j) / sum(R_j) (Eq. 3.13)
        sum_Rj_times_Zj = sum(R * Z for R, Z in zip(firing_strengths_Rj_list, rule_individual_outputs_Zj_list))
        sum_Rj_activations = sum(firing_strengths_Rj_list)
        
        # PT-BR: Evita divisão por zero se nenhuma regra disparar significativamente
        # EN-US: Prevents division by zero when no rule fires significantly
        if sum_Rj_activations < 1e-9:
            # PT-BR: Fallback: usar o K(alpha) do ponto de operação (xb0) mais próximo da POSIÇÃO ATUAL xb_input_val.
            # EN-US: Fallback: use K(alpha) from the operating point (xb0) nearest to the CURRENT POSITION xb_input_val.
            # PT-BR: Este é um mecanismo de segurança; idealmente, as MFs devem cobrir o espaço de operação.
            # EN-US: This is a safety mechanism; ideally, the MFs should cover the operating space.
            op_points_xb0_vals = [op[0] for op in self.operating_points_all]
            closest_op_fallback_idx = np.argmin(np.abs(np.array(op_points_xb0_vals) - xb_input_val))
            K_fallback_gain = self.local_K_alphas_all[closest_op_fallback_idx]
            control_signal_output = -np.dot(K_fallback_gain[0], current_state_vector_phi)
            # PT-BR: Saída opcional para diagnosticar o uso do controlador de contingência.
            # EN-US: Optional output for diagnosing use of the fallback controller.
            # print(f"Aviso: Soma das forças de disparo muito baixa ({sum_Rj_activations:.2e}). Usando fallback Vc: {control_signal_output:.2f} V")
        else:
            control_signal_output = sum_Rj_times_Zj / sum_Rj_activations
            
        return control_signal_output

    def calculate_control_action(self, current_X_states_abs, r_xb_target):
        """PT-BR:
        Calcula a ação de controle fuzzy final Vc.
        current_X_states_abs: vetor de estados atuais [ic, xb, xb_dot, v_int]
        r_xb_target: referência de posição desejada

        EN-US:
        Computes the final fuzzy control action Vc.
        current_X_states_abs: current-state vector [ic, xb, xb_dot, v_int]
        r_xb_target: desired position reference
        """
        ic_current_abs, xb_current_abs, xb_dot_current_abs, v_int_current_abs = current_X_states_abs
        
        # PT-BR: Entradas para as MFs: posição atual e erro atual
        # EN-US: MF inputs: current position and current error
        xb_val_for_mf_input = xb_current_abs
        # PT-BR: Erro: e(t) = r(t) - y(t) (adaptado, pois v_dot na tese é o erro)
        # EN-US: Error: e(t) = r(t) - y(t) (adapted because v_dot is the error in the dissertation)
        v_dot_xb_val_for_mf_input = r_xb_target - xb_current_abs
        
        # PT-BR: Vetor de estados aumentado φ(t) para o consequente da regra
        # EN-US: Augmented state vector φ(t) for the rule consequent
        state_phi_for_consequent_calc = np.array([ic_current_abs, xb_current_abs, xb_dot_current_abs, v_int_current_abs])

        if self.controller_type == "T1":
            # PT-BR: Para T1, há um único caminho de inferência.
            # EN-US: T1 has a single inference path.
            control_action_Vc = self._calculate_single_path_output(
                xb_val_for_mf_input, v_dot_xb_val_for_mf_input, state_phi_for_consequent_calc,
                self.current_mfs_xb_parameters, self.current_mfs_v_dot_xb_parameters
            )
        elif self.controller_type == "T2I":
            # PT-BR: Para T2I, calcula-se a saída para o caminho UMF (u_F1_bar) e LMF (u_F2_underline)
            # EN-US: For T2I, the output is computed for the UMF path (u_F1_bar) and the LMF path (u_F2_underline)
            # PT-BR: e então combina-os (redução de tipo). Figura 3.2 da tese.
            # EN-US: and then combined through type reduction, as shown in dissertation Figure 3.2.
            
            # PT-BR: Caminho Superior (UMF)
            # EN-US: Upper Path (UMF)
            u_F1_bar_output = self._calculate_single_path_output(
                xb_val_for_mf_input, v_dot_xb_val_for_mf_input, state_phi_for_consequent_calc,
                self.mfs_xb_params_umf_set, self.mfs_v_dot_xb_params_umf_set
            )
            # PT-BR: Caminho Inferior (LMF)
            # EN-US: Lower Path (LMF)
            u_F2_underline_output = self._calculate_single_path_output(
                xb_val_for_mf_input, v_dot_xb_val_for_mf_input, state_phi_for_consequent_calc,
                self.mfs_xb_params_lmf_set, self.mfs_v_dot_xb_params_lmf_set
            )
            # PT-BR: Redução de tipo pela média (Eq. 3.14 da tese)
            # EN-US: Type reduction by averaging (dissertation Eq. 3.14)
            control_action_Vc = (u_F1_bar_output + u_F2_underline_output) / 2.0
        else:
            # PT-BR: Não deve acontecer devido à validação no __init__
            # EN-US: This should not occur because of the validation in __init__
            control_action_Vc = 0.0 
        
        # PT-BR: Saturação da tensão de controle (limites físicos do atuador)
        # EN-US: Control-voltage saturation (physical actuator limits)
        return np.clip(control_action_Vc, -self.maglev.V_max, self.maglev.V_max)

print("Componentes do sistema fuzzy (MFs, atribuição de consequentes, classe FuzzyController) definidos.")

In [ ]:
# PT-BR: Definição Detalhada das Funções de Pertinência T2I (Capítulo 4.1.4 da Tese)
# EN-US: Detailed Definition of T2I Membership Functions (Dissertation Chapter 4.1.4)
# PT-BR: Esta seção detalha as Funções de Pertinência Tipo-2 Intervalares (T2I) para o controlador do MAGLEV,
# EN-US: This section details the Interval Type-2 (T2I) membership functions for the MAGLEV controller,
# PT-BR: conforme o Quadro 4.3 e Figuras 4.2 e 4.3 da tese.
# EN-US: following Dissertation Table 4.3 and Figures 4.2 and 4.3.
# PT-BR: O controlador T2I para o MAGLEV na tese utiliza 9 MFs para xb(t) e 9 MFs para v_dot_xb(t).
# EN-US: The dissertation's T2I MAGLEV controller uses 9 MFs for xb(t) and 9 MFs for v_dot_xb(t).
# PT-BR: A FOU (Footprint of Uncertainty) tem larguras entre 0.2mm a 0.5mm.
# EN-US: The FOU (Footprint of Uncertainty) widths range from 0.2 mm to 0.5 mm.

# PT-BR: Nomes das MFs T2I para xb(t) (posição da esfera, em metros) - Conforme Quadro 4.3
# EN-US: Names of the T2I MFs for xb(t) (sphere position, in meters) — according to Table 4.3
mfs_xb_t2i_names_actual = ["x2", "x3.5", "x5", "x6", "x7", "x8", "x9", "x10.5", "x12"]

# PT-BR: Parâmetros para UMFs (Upper Membership Functions) de xb(t) - Extraídos/Interpretados da Figura 4.2
# EN-US: Parameters of the xb(t) UMFs (Upper Membership Functions) — extracted/interpreted from Figure 4.2
# PT-BR: Formato: "NomeMF": ["tipo_mf", [parametros_em_metros]]
# EN-US: Format: "MFName": ["mf_type", [parameters_in_meters]]
# PT-BR: Limites externos da FOU
# EN-US: Outer FOU bounds
mfs_xb_t2i_params_umf_actual = {
    # PT-BR: 0 a 3.5 mm (Estendido)
    # EN-US: 0 to 3.5 mm (extended)
    "x2":   ["trapezoidal", [-1.0, -1.0, 0.00175,0.0035]],
    # PT-BR: 2 a 5 mm
    # EN-US: 2 to 5 mm
    "x3.5": ["triangular",  [0.0020, 0.0035, 0.0050]],
    # PT-BR: 3.5 a 6 mm
    # EN-US: 3.5 to 6 mm
    "x5":   ["triangular",  [0.0035, 0.0050, 0.0060]],
    # PT-BR: 5 a 7 mm
    # EN-US: 5 to 7 mm
    "x6":   ["triangular",  [0.0050, 0.0060, 0.0070]],
    # PT-BR: 6 a 8 mm
    # EN-US: 6 to 8 mm
    "x7":   ["triangular",  [0.0060, 0.0070, 0.0080]],
    # PT-BR: 7 a 9 mm
    # EN-US: 7 to 9 mm
    "x8":   ["triangular",  [0.0070, 0.0080, 0.0090]],
    # PT-BR: 8 a 10.5 mm
    # EN-US: 8 to 10.5 mm
    "x9":   ["triangular",  [0.0080, 0.0090, 0.0105]],
    # PT-BR: 9 a 12 mm
    # EN-US: 9 to 12 mm
    "x10.5":["triangular",  [0.0090, 0.0105, 0.0120]],
    # PT-BR: 10.5 a 14 mm (Estendido)
    # EN-US: 10.5 to 14 mm (extended)
    "x12":  ["trapezoidal", [0.0105, 0.01225, 1.0, 1.0]],
}

# PT-BR: Parâmetros para LMFs (Lower Membership Functions) de xb(t) - Extraídos/Interpretados da Figura 4.2
# EN-US: Parameters of the xb(t) LMFs (Lower Membership Functions) — extracted/interpreted from Figure 4.2
# PT-BR: A LMF é mais "estreita" (interna) que a UMF, definindo a FOU.
# EN-US: The LMF is narrower (inner) than the UMF and therefore defines the FOU.
# PT-BR: FOU de aprox. 0.2mm a 0.5mm de largura total (0.1mm a 0.25mm para cada lado do "centro").
# EN-US: Approximate total FOU width from 0.2 mm to 0.5 mm (0.1 mm to 0.25 mm on each side of the center).
# PT-BR: Exemplo: para x2, FOU ~0.5mm na "rampa". Para x5, FOU ~0.2mm.
# EN-US: Example: x2 has an approximately 0.5 mm FOU on its ramp, whereas x5 has an approximately 0.2 mm FOU.
# PT-BR: Limites internos da FOU
# EN-US: Inner FOU bounds
mfs_xb_t2i_params_lmf_actual = {
    # PT-BR: FOU de 0.5mm na rampa (OK: desce antes)
    # EN-US: 0.5 mm FOU on the ramp (correct: decreases earlier)
    "x2":   ["trapezoidal", [-1.0, -1.0, 0.00125,0.0030]],
    # PT-BR: FOU de ~0.2-0.3mm
    # EN-US: Approximately 0.2–0.3 mm FOU
    "x3.5": ["triangular",  [0.0022, 0.0035, 0.0048]],
    # PT-BR: FOU de ~0.2-0.3mm
    # EN-US: Approximately 0.2–0.3 mm FOU
    "x5":   ["triangular",  [0.0037, 0.0050, 0.0058]],
    # PT-BR: FOU de ~0.2mm
    # EN-US: Approximately 0.2 mm FOU
    "x6":   ["triangular",  [0.0051, 0.0060, 0.0069]],
    # PT-BR: FOU de ~0.2mm
    # EN-US: Approximately 0.2 mm FOU
    "x7":   ["triangular",  [0.0061, 0.0070, 0.0079]],
    # PT-BR: FOU de ~0.2mm
    # EN-US: Approximately 0.2 mm FOU
    "x8":   ["triangular",  [0.0071, 0.0080, 0.0089]],
    # PT-BR: FOU de ~0.2-0.3mm
    # EN-US: Approximately 0.2–0.3 mm FOU
    "x9":   ["triangular",  [0.0082, 0.0090, 0.0103]],
    # PT-BR: FOU de ~0.2-0.3mm
    # EN-US: Approximately 0.2–0.3 mm FOU
    "x10.5":["triangular",  [0.0092, 0.0105, 0.0118]],
    # PT-BR: Sobe depois (10.8) e chega no topo DEPOIS (12.75)
    # EN-US: Rises later (10.8) and reaches the plateau LATER (12.75)
    "x12":  ["trapezoidal", [0.0108, 0.01275, 1.0, 1.0]],
}
# PT-BR: Agrupa UMF e LMF para o controlador T2I
# EN-US: Groups the UMF and LMF definitions for the T2I controller
t2i_xb_params_actual = {'umf': mfs_xb_t2i_params_umf_actual, 'lmf': mfs_xb_t2i_params_lmf_actual}


# PT-BR: Nomes das MFs T2I para v_dot_xb(t) (erro de posição, em metros) - Conforme Quadro 4.3
# EN-US: Names of the T2I MFs for v_dot_xb(t) (position error, in meters) — according to Table 4.3
# PT-BR: A Figura 4.3 usa "eZ" para erro zero.
# EN-US: Figure 4.3 uses "eZ" for zero error.
mfs_v_dot_xb_t2i_names_actual = ["Nmax", "Ng", "Nm", "Np", "eZ", "Pp", "Pm", "Pg", "Pmax"]

# PT-BR: Parâmetros para UMFs de v_dot_xb(t) - Extraídos/Interpretados da Figura 4.3
# EN-US: Parameters of the v_dot_xb(t) UMFs — extracted/interpreted from Figure 4.3
# PT-BR: Universo de discurso para erro na Figura 4.3 vai de aprox. -14mm a +14mm.
# EN-US: The error universe of discourse in Figure 4.3 spans approximately -14 mm to +14 mm.
# PT-BR: Metade do range principal da Figura 4.3 (ex: +/-7mm), com extensões
# EN-US: Half of the main range in Figure 4.3 (for example, +/- 7 mm), with extensions
err_lim_t2i_fig = 0.007
mfs_v_dot_xb_t2i_params_umf_actual = {
    # PT-BR: -14mm a -7mm (Estendido)
    # EN-US: -14 mm to -7 mm (extended)
    "Nmax": ["trapezoidal", [-1.0, -1.0, -1.5*err_lim_t2i_fig, -1.0*err_lim_t2i_fig]],
    # PT-BR: Triangular
    # EN-US: Triangular
    "Ng":   ["triangular", [-1.5*err_lim_t2i_fig, -1.0*err_lim_t2i_fig, -0.6*err_lim_t2i_fig]],
    # PT-BR: Ex: cobrindo -4mm
    # EN-US: For example, covers -4 mm
    "Nm":   ["triangular",  [-1.0*err_lim_t2i_fig, -0.6*err_lim_t2i_fig, -0.25*err_lim_t2i_fig]],
    # PT-BR: Ex: cobrindo -1.5mm
    # EN-US: For example, covers -1.5 mm
    "Np":   ["triangular",  [-0.6*err_lim_t2i_fig, -0.25*err_lim_t2i_fig, 0.0*err_lim_t2i_fig]],
    # PT-BR: +/-1mm (aprox)
    # EN-US: +/- 1 mm (approximately)
    "eZ":   ["triangular",  [-0.15*err_lim_t2i_fig, 0.0*err_lim_t2i_fig,  0.15*err_lim_t2i_fig]],
    "Pp":   ["triangular",  [0.0*err_lim_t2i_fig,   0.25*err_lim_t2i_fig, 0.6*err_lim_t2i_fig]],
    "Pm":   ["triangular",  [0.25*err_lim_t2i_fig,  0.6*err_lim_t2i_fig,  1.0*err_lim_t2i_fig]],
    # PT-BR: Triangular
    # EN-US: Triangular
    "Pg":   ["triangular", [0.6*err_lim_t2i_fig,   1.0*err_lim_t2i_fig,  1.5*err_lim_t2i_fig]],
    # PT-BR: 7mm a 14mm (Estendido)
    # EN-US: 7 mm to 14 mm (extended)
    "Pmax": ["trapezoidal", [1.0*err_lim_t2i_fig,   1.5*err_lim_t2i_fig,  1.0,  1.0]],
}

# PT-BR: Parâmetros para LMFs de v_dot_xb(t) - Extraídos/Interpretados da Figura 4.3 (FOU de ~0.2mm a 0.5mm)
# EN-US: Parameters of the v_dot_xb(t) LMFs — extracted/interpreted from Figure 4.3 (FOU of about 0.2–0.5 mm)
# PT-BR: A FOU é aplicada deslocando os pontos das MFs UMF.
# EN-US: The FOU is applied by shifting the points of the UMFs.
# PT-BR: Exemplo: FOU de 0.4mm (0.2mm para cada lado do "centro" da MF)
# EN-US: Example: 0.4 mm FOU (0.2 mm on each side of the MF center)
# PT-BR: 0.2mm
# EN-US: 0.2 mm
fou_half_width_v_dot = 0.0002
mfs_v_dot_xb_t2i_params_lmf_actual = {
    # PT-BR: Deslocar para esquerda (-fou)
    # EN-US: Shift to the left (-fou)
    "Nmax": ["trapezoidal", [-1.0+fou_half_width_v_dot, -1.0+fou_half_width_v_dot, -1.5*err_lim_t2i_fig-fou_half_width_v_dot, -1.0*err_lim_t2i_fig-fou_half_width_v_dot]],
    "Ng":   ["triangular",  [-1.5*err_lim_t2i_fig+fou_half_width_v_dot, -1.0*err_lim_t2i_fig, -0.6*err_lim_t2i_fig-fou_half_width_v_dot]],
    "Nm":   ["triangular",  [-1.0*err_lim_t2i_fig+fou_half_width_v_dot, -0.6*err_lim_t2i_fig, -0.25*err_lim_t2i_fig-fou_half_width_v_dot]],
    # PT-BR: Cuidado ao cruzar zero
    # EN-US: Take care when crossing zero
    "Np":   ["triangular",  [-0.6*err_lim_t2i_fig+fou_half_width_v_dot, -0.25*err_lim_t2i_fig, 0.0*err_lim_t2i_fig-fou_half_width_v_dot]],
    # PT-BR: FOU menor para eZ
    # EN-US: Smaller FOU for eZ
    "eZ":   ["triangular",  [-0.15*err_lim_t2i_fig+fou_half_width_v_dot/2, 0.0*err_lim_t2i_fig,  0.15*err_lim_t2i_fig-fou_half_width_v_dot/2]],
    "Pp":   ["triangular",  [0.0*err_lim_t2i_fig+fou_half_width_v_dot,   0.25*err_lim_t2i_fig, 0.6*err_lim_t2i_fig-fou_half_width_v_dot]],
    "Pm":   ["triangular",  [0.25*err_lim_t2i_fig+fou_half_width_v_dot,  0.6*err_lim_t2i_fig,  1.0*err_lim_t2i_fig-fou_half_width_v_dot]],
    "Pg":   ["triangular",  [0.6*err_lim_t2i_fig+fou_half_width_v_dot,   1.0*err_lim_t2i_fig,  1.5*err_lim_t2i_fig-fou_half_width_v_dot]],
    # PT-BR: Deslocar para direita (+fou)
    # EN-US: Shift to the right (+fou)
    "Pmax": ["trapezoidal", [1.0*err_lim_t2i_fig+fou_half_width_v_dot,   1.5*err_lim_t2i_fig+fou_half_width_v_dot,  1.0-fou_half_width_v_dot,  1.0-fou_half_width_v_dot]],
}

# PT-BR: Ajuste fino para garantir que a <= b (tri) ou a <= b <= c (trap) etc., após aplicar FOU
# EN-US: Fine adjustment to ensure a <= b (tri) or a <= b <= c (trap), etc., after applying the FOU
# PT-BR: Esta é uma simplificação, a extração precisa da figura é ideal.
# EN-US: This is a simplification; precise extraction from the figure would be preferable.
def _ensure_mf_param_order(params_list, mf_type_str):
    # PT-BR: Cópia para modificar
    # EN-US: Copy to be modified
    p = list(params_list)
    # PT-BR: a, b, c
    # EN-US: a, b, c
    if mf_type_str == "triangular":
        if not p[0] <= p[1]: p[0] = p[1] - abs(p[1]*0.01 + 1e-6)
        if not p[1] <= p[2]: p[2] = p[1] + abs(p[1]*0.01 + 1e-6)
        # PT-BR: Garante não inversão
        # EN-US: Prevents inversion
        if p[0] > p[1] : p[0] = p[1]
        if p[2] < p[1] : p[2] = p[1]
    # PT-BR: a, b, c, d
    # EN-US: a, b, c, d
    elif mf_type_str == "trapezoidal":
        if not p[0] <= p[1]: p[0] = p[1] - abs(p[1]*0.01 + 1e-6)
        # PT-BR: b pode mover para trás de c
        # EN-US: b may move behind c
        if not p[1] <= p[2]: p[1] = p[2] - abs(p[2]*0.01 + 1e-6)
        if not p[2] <= p[3]: p[3] = p[2] + abs(p[2]*0.01 + 1e-6)
        # PT-BR: Re-checa e ajusta
        # EN-US: Checks again and adjusts
        if p[0] > p[1]: p[0] = p[1]
        if p[1] > p[2]: p[1] = p[2]
        if p[2] > p[3]: p[2] = p[3]
    return tuple(p)

for name_key in mfs_v_dot_xb_t2i_params_lmf_actual:
    mf_type, params_val = mfs_v_dot_xb_t2i_params_lmf_actual[name_key]
    mfs_v_dot_xb_t2i_params_lmf_actual[name_key][1] = _ensure_mf_param_order(params_val, mf_type)

t2i_v_dot_xb_params_actual = {'umf': mfs_v_dot_xb_t2i_params_umf_actual, 'lmf': mfs_v_dot_xb_t2i_params_lmf_actual}

print("Definidas MFs T2I (UMF e LMF) para xb(t) e v_dot_xb(t) conforme interpretação da tese.")

In [ ]:
# PT-BR: === SIMULAÇÃO DO SISTEMA MAGLEV COM CONTROLADORES OTIMIZADOS (MULTI-BITS) ===
# EN-US: === MAGLEV SYSTEM SIMULATION WITH OPTIMIZED CONTROLLERS (MULTIPLE BIT RESOLUTIONS) ===

if 'maglev' not in locals():
    maglev = MaglevSystem()
    local_controllers_K_list = design_local_controllers(maglev, operating_points)

# PT-BR: --- Parâmetros da Simulação ---
# EN-US: --- Simulation Parameters ---
sim_t_final       = 10.0
sim_t_final_full  = 10.0
sim_dt_controller = 0.002

idx_op_inicial_sim = 3
sim_initial_xb = operating_points[idx_op_inicial_sim][0]
sim_initial_ic = operating_points[idx_op_inicial_sim][1]
sim_X0 = np.array([sim_initial_ic, sim_initial_xb, 0.0, 0.0])

def reference_signal_sim(t_current):
    if t_current < 0.5:  return sim_initial_xb
    if t_current < 2.5:  return 0.005
    elif t_current < 5.0: return 0.009
    elif t_current < 7.5: return 0.003
    else:                 return 0.007

# =============================================================================
# PT-BR: CONTROLADORES ORIGINAIS (apenas 1 vez - independente dos bits)
# EN-US: ORIGINAL CONTROLLERS (only once — independent of the bit resolution)
# =============================================================================
print("\n" + "="*70)
print("Re-inicializando Controladores Fuzzy Originais...")
print("="*70)

fuzzy_controller_T1_sim = FuzzyController(
    maglev, local_controllers_K_list, operating_points,
    mfs_xb_t1_names, mfs_xb_t1_params,
    mfs_v_dot_xb_t1_names, mfs_v_dot_xb_t1_params,
    controller_type_str="T1"
)
fuzzy_controller_T2I_actual_sim = FuzzyController(
    maglev, local_controllers_K_list, operating_points,
    mfs_xb_t2i_names_actual, t2i_xb_params_actual,
    mfs_v_dot_xb_t2i_names_actual, t2i_v_dot_xb_params_actual,
    controller_type_str="T2I"
)

# PT-BR: Simulações originais (sem otimização)
# EN-US: Original simulations (without optimization)
if 'data_log_T1' not in globals() or data_log_T1 is None:
    print("\nExecutando simulação T1 Original...")
    data_log_T1 = run_simulation(maglev, fuzzy_controller_T1_sim,
                                 sim_X0.copy(), sim_t_final,
                                 sim_dt_controller, reference_signal_sim)
else:
    print("data_log_T1 já carregado. Pulando simulação original T1.")

if 'data_log_T2I' not in globals() or data_log_T2I is None:
    print("\nExecutando simulação T2I Original...")
    data_log_T2I = run_simulation(maglev, fuzzy_controller_T2I_actual_sim,
                                  sim_X0.copy(), sim_t_final,
                                  sim_dt_controller, reference_signal_sim)
else:
    print("data_log_T2I já carregado. Pulando simulação original T2I.")

# =============================================================================
# PT-BR: CONTROLADORES GA STANDARD (apenas 1 vez - igual para todos os bits)
# EN-US: STANDARD GA CONTROLLERS (only once — identical for every bit resolution)
# =============================================================================
print("\n" + "="*70)
print("Preparando Controladores GA Standard Otimizados...")
print("="*70)

if 'optimized_desired_poles_T1' in globals() and optimized_desired_poles_T1 is not None:
    if 'data_log_T1_opt' not in globals() or data_log_T1_opt is None:
        print("\nProjetando K(alpha) para T1 Otimizado (GA Standard)...")
        local_controllers_K_list_opt_T1 = design_local_controllers_modifiable_poles(
            maglev, operating_points, optimized_desired_poles_T1)

        print("Inicializando Controlador Fuzzy T1 Otimizado (GA Standard)...")
        fuzzy_controller_T1_optimized_sim = FuzzyController(
            maglev, local_controllers_K_list_opt_T1, operating_points,
            mfs_xb_t1_names, mfs_xb_t1_params,
            mfs_v_dot_xb_t1_names, mfs_v_dot_xb_t1_params,
            controller_type_str="T1"
        )
        print("Executando simulação T1 Otimizado (GA Standard)...")
        data_log_T1_opt = run_simulation(maglev, fuzzy_controller_T1_optimized_sim,
                                         sim_X0.copy(), sim_t_final_full,
                                         sim_dt_controller, reference_signal_sim)
    else:
        print("data_log_T1_opt já carregado. Pulando simulação GA T1.")
else:
    print("AVISO: optimized_desired_poles_T1 não disponível. GA Standard T1 ignorado.")
    data_log_T1_opt = None

if 'optimized_desired_poles_T2I' in globals() and optimized_desired_poles_T2I is not None:
    if 'data_log_T2I_opt' not in globals() or data_log_T2I_opt is None:
        print("\nProjetando K(alpha) para T2I Otimizado (GA Standard)...")
        local_controllers_K_list_opt_T2I = design_local_controllers_modifiable_poles(
            maglev, operating_points, optimized_desired_poles_T2I)

        print("Inicializando Controlador Fuzzy T2I Otimizado (GA Standard)...")
        fuzzy_controller_T2I_optimized_sim = FuzzyController(
            maglev, local_controllers_K_list_opt_T2I, operating_points,
            mfs_xb_t2i_names_actual, t2i_xb_params_actual,
            mfs_v_dot_xb_t2i_names_actual, t2i_v_dot_xb_params_actual,
            controller_type_str="T2I"
        )
        print("Executando simulação T2I Otimizado (GA Standard)...")
        data_log_T2I_opt = run_simulation(maglev, fuzzy_controller_T2I_optimized_sim,
                                          sim_X0.copy(), sim_t_final_full,
                                          sim_dt_controller, reference_signal_sim)
    else:
        print("data_log_T2I_opt já carregado. Pulando simulação GA T2I.")
else:
    print("AVISO: optimized_desired_poles_T2I não disponível. GA Standard T2I ignorado.")
    data_log_T2I_opt = None

# =============================================================================
# PT-BR: CONTROLADORES QGA - LOOP PELOS 4 ARQUIVOS (10, 16, 24, 32 bits)
# EN-US: QGA CONTROLLERS — LOOP OVER THE 4 FILES (10, 16, 24, 32 bits)
# =============================================================================
print("\n" + "="*70)
print("Preparando Controladores QGA Otimizados para cada resolução de bits...")
print("="*70)

# PT-BR: Dicionários globais para armazenar resultados por bits
# EN-US: Global dictionaries that store results by bit resolution
# PT-BR: data_logs_qga_T1[bits]  = data_log
# EN-US: Expected structure for storing the T1 log for each resolution
data_logs_qga_T1  = {}
# PT-BR: data_logs_qga_T2I[bits] = data_log
# EN-US: Expected structure for storing the T2I log for each resolution
data_logs_qga_T2I = {}
# PT-BR: polos_qga_T1[bits]      = array de polos
# EN-US: Expected structure for storing the T1 poles for each resolution
polos_qga_T1      = {}
# PT-BR: polos_qga_T2I[bits]     = array de polos
# EN-US: Expected structure for storing the T2I poles for each resolution
polos_qga_T2I     = {}

for bits in bits_list:
    dados_bits = all_dados.get(bits)

    print(f"\n{'─'*60}")
    print(f"  Processando QGA {bits} bits...")
    print(f"{'─'*60}")

    if dados_bits is None:
        print(f"  AVISO: Dados de {bits} bits não carregados. Pulando.")
        data_logs_qga_T1[bits]  = None
        data_logs_qga_T2I[bits] = None
        polos_qga_T1[bits]      = None
        polos_qga_T2I[bits]     = None
        continue

    # PT-BR: --- Extrai variáveis específicas deste arquivo ---
    # EN-US: --- Extracts variables specific to this file ---
    poles_T1_bits  = dados_bits.get('qga_optimized_desired_poles_T1_qga')
    poles_T2I_bits = dados_bits.get('qga_optimized_desired_poles_T2I_qga')

    # PT-BR: Tenta log já salvo no arquivo (se existir)
    # EN-US: Attempts to use a log already stored in the file (if available)
    log_T1_bits  = dados_bits.get('qga_data_log_T1_opt_qga')
    log_T2I_bits = dados_bits.get('qga_data_log_T2I_opt_qga')

    polos_qga_T1[bits]  = poles_T1_bits
    polos_qga_T2I[bits] = poles_T2I_bits

    # ------------------------------------------------------------------
    # PT-BR: T1 QGA
    # EN-US: T1 QGA
    # ------------------------------------------------------------------
    if isinstance(log_T1_bits, dict) and 't' in log_T1_bits:
        print(f"  OK: Log T1 QGA {bits}bits já salvo no PKL "
              f"({len(log_T1_bits['t'])} pontos). Pulando simulação.")
        data_logs_qga_T1[bits] = log_T1_bits
    elif poles_T1_bits is not None:
        print(f"  Projetando K(alpha) para T1 QGA {bits}bits...")
        print(f"    Polos: {poles_T1_bits}")
        K_list_T1 = design_local_controllers_modifiable_poles(
            maglev, operating_points, poles_T1_bits)

        if K_list_T1 is not None:
            print(f"  Inicializando Controlador Fuzzy T1 QGA {bits}bits...")
            ctrl_T1 = FuzzyController(
                maglev, K_list_T1, operating_points,
                mfs_xb_t1_names, mfs_xb_t1_params,
                mfs_v_dot_xb_t1_names, mfs_v_dot_xb_t1_params,
                controller_type_str="T1"
            )
            print(f"  Executando simulação T1 QGA {bits}bits...")
            data_logs_qga_T1[bits] = run_simulation(
                maglev, ctrl_T1, sim_X0.copy(),
                sim_t_final_full, sim_dt_controller, reference_signal_sim
            )
        else:
            print(f"  ERRO: Falha ao projetar K(alpha) para T1 QGA {bits}bits.")
            data_logs_qga_T1[bits] = None
    else:
        print(f"  AVISO: Polos T1 QGA {bits}bits não encontrados no arquivo.")
        data_logs_qga_T1[bits] = None

    # ------------------------------------------------------------------
    # PT-BR: T2I QGA
    # EN-US: T2I QGA
    # ------------------------------------------------------------------
    if isinstance(log_T2I_bits, dict) and 't' in log_T2I_bits:
        print(f"  OK: Log T2I QGA {bits}bits já salvo no PKL "
              f"({len(log_T2I_bits['t'])} pontos). Pulando simulação.")
        data_logs_qga_T2I[bits] = log_T2I_bits
    elif poles_T2I_bits is not None:
        print(f"  Projetando K(alpha) para T2I QGA {bits}bits...")
        print(f"    Polos: {poles_T2I_bits}")
        K_list_T2I = design_local_controllers_modifiable_poles(
            maglev, operating_points, poles_T2I_bits)

        if K_list_T2I is not None:
            print(f"  Inicializando Controlador Fuzzy T2I QGA {bits}bits...")
            ctrl_T2I = FuzzyController(
                maglev, K_list_T2I, operating_points,
                mfs_xb_t2i_names_actual, t2i_xb_params_actual,
                mfs_v_dot_xb_t2i_names_actual, t2i_v_dot_xb_params_actual,
                controller_type_str="T2I"
            )
            print(f"  Executando simulação T2I QGA {bits}bits...")
            data_logs_qga_T2I[bits] = run_simulation(
                maglev, ctrl_T2I, sim_X0.copy(),
                sim_t_final_full, sim_dt_controller, reference_signal_sim
            )
        else:
            print(f"  ERRO: Falha ao projetar K(alpha) para T2I QGA {bits}bits.")
            data_logs_qga_T2I[bits] = None
    else:
        print(f"  AVISO: Polos T2I QGA {bits}bits não encontrados no arquivo.")
        data_logs_qga_T2I[bits] = None

# =============================================================================
# PT-BR: Compatibilidade com células posteriores (variáveis sem sufixo de bits)
# EN-US: Compatibility with later cells (variables without a bit-resolution suffix)
# PT-BR: Usa o primeiro bits carregado com sucesso como referência
# EN-US: Uses the first bit resolution loaded successfully as the reference
# =============================================================================
bits_com_log_T1  = [b for b in bits_list if data_logs_qga_T1.get(b)  is not None]
bits_com_log_T2I = [b for b in bits_list if data_logs_qga_T2I.get(b) is not None]

if bits_com_log_T1:
    data_log_T1_opt_qga             = data_logs_qga_T1[bits_com_log_T1[0]]
    optimized_desired_poles_T1_qga  = polos_qga_T1[bits_com_log_T1[0]]
    print(f"\nReferência T1 QGA → {bits_com_log_T1[0]} bits "
          f"(mapeado para 'data_log_T1_opt_qga')")

if bits_com_log_T2I:
    data_log_T2I_opt_qga            = data_logs_qga_T2I[bits_com_log_T2I[0]]
    optimized_desired_poles_T2I_qga = polos_qga_T2I[bits_com_log_T2I[0]]
    print(f"Referência T2I QGA → {bits_com_log_T2I[0]} bits "
          f"(mapeado para 'data_log_T2I_opt_qga')")

# =============================================================================
# PT-BR: Resumo Final
# EN-US: Final Summary
# =============================================================================
print("\n" + "="*70)
print("RESUMO DAS SIMULAÇÕES")
print("="*70)
print(f"  T1 Original:      {'OK:' if data_log_T1  is not None else 'ERRO:'}")
print(f"  T2I Original:     {'OK:' if data_log_T2I is not None else 'ERRO:'}")
print(f"  T1 GA Standard:   {'OK:' if data_log_T1_opt  is not None else 'ERRO:'}")
print(f"  T2I GA Standard:  {'OK:' if data_log_T2I_opt is not None else 'ERRO:'}")
for bits in bits_list:
    st1  = 'OK:' if data_logs_qga_T1 .get(bits) is not None else 'ERRO:'
    st2i = 'OK:' if data_logs_qga_T2I.get(bits) is not None else 'ERRO:'
    print(f"  QGA {bits:2}bits: T1={st1} | T2I={st2i}  "
          f"| Polos T1={polos_qga_T1.get(bits)}  "
          f"| Polos T2I={polos_qga_T2I.get(bits)}")
print("="*70)
print("\nAcesse os logs via:")
print("  data_logs_qga_T1[10], data_logs_qga_T1[16], data_logs_qga_T1[24], data_logs_qga_T1[32]")
print("  data_logs_qga_T2I[10], data_logs_qga_T2I[16], data_logs_qga_T2I[24], data_logs_qga_T2I[32]")

In [ ]:
# =============================================================================
# PT-BR: PLOTAGEM DOS RESULTADOS - ORIGINAIS + GA STANDARD + QGA (10, 16, 24, 32 bits)
# EN-US: RESULT PLOTS — ORIGINAL + STANDARD GA + QGA (10, 16, 24, 32 bits)
# =============================================================================
plt.style.use('seaborn-v0_8-whitegrid')

# PT-BR: --- Paletas de cores por bits ---
# EN-US: --- Color palettes by bit resolution ---
cores_qga = {10: 'crimson', 16: 'darkorange', 24: 'purple', 32: 'saddlebrown'}
ls_qga    = {10: '-',       16: '-.',          24: (0,(3,1,1,1)), 32: ':'}

# =============================================================================
# PT-BR: BLOCO 1 - Comparação Somente Originais (T1 vs T2I)
# EN-US: BLOCK 1 — Original-Controller Comparison Only (T1 vs. T2I)
# =============================================================================
for var_name, titulo in [
    ('xb',  'Posição da Esfera $x_b(t)$ - Originais'),
    ('Vc',  'Tensão de Controle $V_c(t)$ - Originais'),
    ('ic',  'Corrente na Bobina $i_c(t)$ - Originais'),
]:
    plt.figure(figsize=(14, 5))
    if data_log_T1 is not None:
        plt.plot(data_log_T1['t'], data_log_T1[var_name] * (1000 if var_name == 'xb' else 1),
                 label='T1 Original', color='blue', linewidth=2)
    if data_log_T2I is not None:
        plt.plot(data_log_T2I['t'], data_log_T2I[var_name] * (1000 if var_name == 'xb' else 1),
                 label='T2I Original', color='green', linestyle='--', linewidth=2)
    if var_name == 'xb' and data_log_T1 is not None:
        plt.plot(data_log_T1['t'], data_log_T1['r_xb'] * 1000,
                 label='Referência $r(t)$', linestyle=':', color='k', linewidth=2)
    ylabel_map = {'xb': 'Posição (mm)', 'Vc': 'Tensão (V)', 'ic': 'Corrente (A)'}
    plt.ylabel(ylabel_map[var_name])
    plt.xlabel('Tempo (s)')
    plt.title(titulo)
    plt.legend()
    plt.grid(True)
    if var_name == 'xb':
        plt.ylim(0, maglev.x_b_max * 1000 + 1)
    elif var_name == 'Vc':
        plt.ylim(-maglev.V_max - 1, maglev.V_max + 1)
    plt.tight_layout()
    plt.show()

# =============================================================================
# PT-BR: BLOCO 2 - Subplots individuais T1 Original (3 painéis)
# EN-US: BLOCK 2 — Individual T1 Original subplots (3 panels)
# =============================================================================
if data_log_T1 is not None:
    fig_t1, axs_t1 = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    fig_t1.suptitle('Resultados da Simulação MAGLEV - Controlador Fuzzy T1 Original', fontsize=14)

    axs_t1[0].plot(data_log_T1['t'], data_log_T1['xb'] * 1000, label='Posição Esfera $x_b(t)$', color='blue')
    axs_t1[0].plot(data_log_T1['t'], data_log_T1['r_xb'] * 1000, label='Referência $r(t)$', linestyle=':', color='k')
    axs_t1[0].set_ylabel('Posição (mm)')
    axs_t1[0].legend()
    axs_t1[0].grid(True)
    axs_t1[0].set_ylim(0, maglev.x_b_max * 1000 + 1)

    axs_t1[1].plot(data_log_T1['t'], data_log_T1['Vc'], label='Tensão Controle $V_c(t)$', color='r')
    axs_t1[1].set_ylabel('Tensão (V)')
    axs_t1[1].legend()
    axs_t1[1].grid(True)
    axs_t1[1].set_ylim(-maglev.V_max - 1, maglev.V_max + 1)

    axs_t1[2].plot(data_log_T1['t'], data_log_T1['ic'], label='Corrente Bobina $i_c(t)$', color='g')
    axs_t1[2].set_xlabel('Tempo (s)')
    axs_t1[2].set_ylabel('Corrente (A)')
    axs_t1[2].legend()
    axs_t1[2].grid(True)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# =============================================================================
# PT-BR: BLOCO 3 - Subplots individuais T2I Original (3 painéis)
# EN-US: BLOCK 3 — Individual T2I Original subplots (3 panels)
# =============================================================================
if data_log_T2I is not None:
    fig_t2i, axs_t2i = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    fig_t2i.suptitle('Resultados da Simulação MAGLEV - Controlador Fuzzy T2I Original', fontsize=14)

    axs_t2i[0].plot(data_log_T2I['t'], data_log_T2I['xb'] * 1000, label='Posição Esfera $x_b(t)$', color='green')
    axs_t2i[0].plot(data_log_T2I['t'], data_log_T2I['r_xb'] * 1000, label='Referência $r(t)$', linestyle=':', color='k')
    axs_t2i[0].set_ylabel('Posição (mm)')
    axs_t2i[0].legend()
    axs_t2i[0].grid(True)
    axs_t2i[0].set_ylim(0, maglev.x_b_max * 1000 + 1)

    axs_t2i[1].plot(data_log_T2I['t'], data_log_T2I['Vc'], label='Tensão Controle $V_c(t)$', color='m')
    axs_t2i[1].set_ylabel('Tensão (V)')
    axs_t2i[1].legend()
    axs_t2i[1].grid(True)
    axs_t2i[1].set_ylim(-maglev.V_max - 1, maglev.V_max + 1)

    axs_t2i[2].plot(data_log_T2I['t'], data_log_T2I['ic'], label='Corrente Bobina $i_c(t)$', color='c')
    axs_t2i[2].set_xlabel('Tempo (s)')
    axs_t2i[2].set_ylabel('Corrente (A)')
    axs_t2i[2].legend()
    axs_t2i[2].grid(True)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# =============================================================================
# PT-BR: BLOCO 4 - Comparação Geral: Originais + GA Standard + QGA (todos os bits)
# EN-US: BLOCK 4 — Overall Comparison: Original + Standard GA + QGA (all bit resolutions)
# PT-BR: Posição da Esfera
# EN-US: Sphere Position
# =============================================================================
print("\nGerando gráficos comparativos completos (Original + GA + QGA multi-bits)...")

for tipo_ctrl in ['T1', 'T2I']:
    fig, ax = plt.subplots(figsize=(16, 7))

    # PT-BR: --- Original ---
    # EN-US: --- Original ---
    log_orig = data_log_T1 if tipo_ctrl == 'T1' else data_log_T2I
    if log_orig is not None:
        ax.plot(log_orig['t'], log_orig['xb'] * 1000,
                label=f'{tipo_ctrl} Original',
                color='gray', linestyle=':', linewidth=2, alpha=0.9)
        # PT-BR: Referência (usa o log original como base de tempo)
        # EN-US: Reference (uses the original log as the time base)
        ax.plot(log_orig['t'], log_orig['r_xb'] * 1000,
                label='Referência $r(t)$',
                color='k', linestyle='--', linewidth=2, alpha=0.7)

    # PT-BR: --- GA Standard ---
    # EN-US: --- Standard GA ---
    log_ga = data_log_T1_opt if tipo_ctrl == 'T1' else data_log_T2I_opt
    if log_ga is not None:
        ax.plot(log_ga['t'], log_ga['xb'] * 1000,
                label=f'{tipo_ctrl} GA Standard',
                color='deepskyblue' if tipo_ctrl == 'T1' else 'limegreen',
                linestyle='-.', linewidth=2)

    # PT-BR: --- QGA por bits ---
    # EN-US: --- QGA by bit resolution ---
    logs_qga = data_logs_qga_T1 if tipo_ctrl == 'T1' else data_logs_qga_T2I
    for bits in bits_list:
        log_bits = logs_qga.get(bits)
        if log_bits is not None:
            ax.plot(log_bits['t'], log_bits['xb'] * 1000,
                    label=f'{tipo_ctrl} QGA {bits}bits',
                    color=cores_qga[bits],
                    linestyle=ls_qga[bits],
                    linewidth=2)

    ax.set_ylabel('Posição da Esfera (mm)')
    ax.set_xlabel('Tempo (s)')
    ax.set_title(f'Comparação de Posição - Controlador {tipo_ctrl}: Original vs GA vs QGA (multi-bits)')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True)
    ax.set_ylim(0, maglev.x_b_max * 1000 + 1)
    plt.tight_layout()
    plt.show()

# =============================================================================
# PT-BR: BLOCO 5 - Comparação Geral: Tensão de Controle
# EN-US: BLOCK 5 — Overall Comparison: Control Voltage
# =============================================================================
for tipo_ctrl in ['T1', 'T2I']:
    fig, ax = plt.subplots(figsize=(16, 5))

    log_orig = data_log_T1 if tipo_ctrl == 'T1' else data_log_T2I
    if log_orig is not None:
        ax.plot(log_orig['t'], log_orig['Vc'],
                label=f'{tipo_ctrl} Original', color='gray', linestyle=':', linewidth=2, alpha=0.9)

    log_ga = data_log_T1_opt if tipo_ctrl == 'T1' else data_log_T2I_opt
    if log_ga is not None:
        ax.plot(log_ga['t'], log_ga['Vc'],
                label=f'{tipo_ctrl} GA Standard',
                color='deepskyblue' if tipo_ctrl == 'T1' else 'limegreen',
                linestyle='-.', linewidth=2)

    logs_qga = data_logs_qga_T1 if tipo_ctrl == 'T1' else data_logs_qga_T2I
    for bits in bits_list:
        log_bits = logs_qga.get(bits)
        if log_bits is not None:
            ax.plot(log_bits['t'], log_bits['Vc'],
                    label=f'{tipo_ctrl} QGA {bits}bits',
                    color=cores_qga[bits],
                    linestyle=ls_qga[bits],
                    linewidth=2, alpha=0.85)

    ax.set_ylabel('Tensão de Controle (V)')
    ax.set_xlabel('Tempo (s)')
    ax.set_title(f'Comparação de Tensão de Controle - Controlador {tipo_ctrl}')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True)
    ax.set_ylim(-maglev.V_max - 1, maglev.V_max + 1)
    plt.tight_layout()
    plt.show()

# =============================================================================
# PT-BR: BLOCO 6 - Comparação Geral: Corrente na Bobina
# EN-US: BLOCK 6 — Overall Comparison: Coil Current
# =============================================================================
for tipo_ctrl in ['T1', 'T2I']:
    fig, ax = plt.subplots(figsize=(16, 5))

    log_orig = data_log_T1 if tipo_ctrl == 'T1' else data_log_T2I
    if log_orig is not None:
        ax.plot(log_orig['t'], log_orig['ic'],
                label=f'{tipo_ctrl} Original', color='gray', linestyle=':', linewidth=2, alpha=0.9)

    log_ga = data_log_T1_opt if tipo_ctrl == 'T1' else data_log_T2I_opt
    if log_ga is not None:
        ax.plot(log_ga['t'], log_ga['ic'],
                label=f'{tipo_ctrl} GA Standard',
                color='deepskyblue' if tipo_ctrl == 'T1' else 'limegreen',
                linestyle='-.', linewidth=2)

    logs_qga = data_logs_qga_T1 if tipo_ctrl == 'T1' else data_logs_qga_T2I
    for bits in bits_list:
        log_bits = logs_qga.get(bits)
        if log_bits is not None:
            ax.plot(log_bits['t'], log_bits['ic'],
                    label=f'{tipo_ctrl} QGA {bits}bits',
                    color=cores_qga[bits],
                    linestyle=ls_qga[bits],
                    linewidth=2, alpha=0.85)

    ax.set_ylabel('Corrente na Bobina (A)')
    ax.set_xlabel('Tempo (s)')
    ax.set_title(f'Comparação de Corrente na Bobina - Controlador {tipo_ctrl}')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True)
    plt.tight_layout()
    plt.show()

# =============================================================================
# PT-BR: BLOCO 7 - Subplots individuais por bits QGA (3 painéis: pos, Vc, ic)
# EN-US: BLOCK 7 — Individual QGA subplots by bit resolution (3 panels: position, Vc, ic)
# =============================================================================
for tipo_ctrl in ['T1', 'T2I']:
    logs_qga = data_logs_qga_T1 if tipo_ctrl == 'T1' else data_logs_qga_T2I
    for bits in bits_list:
        log_bits = logs_qga.get(bits)
        if log_bits is None:
            print(f"  AVISO: Sem dados para {tipo_ctrl} QGA {bits}bits. Pulando subplot.")
            continue

        fig_b, axs_b = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
        fig_b.suptitle(f'MAGLEV - Controlador Fuzzy {tipo_ctrl} Otimizado (QGA {bits} bits)', fontsize=14)

        axs_b[0].plot(log_bits['t'], log_bits['xb'] * 1000,
                      label=f'Posição $x_b(t)$ - QGA {bits}bits', color=cores_qga[bits], linewidth=2)
        axs_b[0].plot(log_bits['t'], log_bits['r_xb'] * 1000,
                      label='Referência $r(t)$', linestyle=':', color='k', linewidth=2)
        axs_b[0].set_ylabel('Posição (mm)')
        axs_b[0].legend()
        axs_b[0].grid(True)
        axs_b[0].set_ylim(0, maglev.x_b_max * 1000 + 1)

        axs_b[1].plot(log_bits['t'], log_bits['Vc'],
                      label='Tensão Controle $V_c(t)$', color='r', linewidth=2)
        axs_b[1].set_ylabel('Tensão (V)')
        axs_b[1].legend()
        axs_b[1].grid(True)
        axs_b[1].set_ylim(-maglev.V_max - 1, maglev.V_max + 1)

        axs_b[2].plot(log_bits['t'], log_bits['ic'],
                      label='Corrente Bobina $i_c(t)$', color='g', linewidth=2)
        axs_b[2].set_xlabel('Tempo (s)')
        axs_b[2].set_ylabel('Corrente (A)')
        axs_b[2].legend()
        axs_b[2].grid(True)

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

print("\nOK: Plotagem completa finalizada.")

In [ ]:
# PT-BR: --- Plotagem Comparativa (Original vs Otimizado) ---
# EN-US: --- Comparative Plot (Original vs. Optimized) ---
plt.style.use('seaborn-v0_8-whitegrid')

# PT-BR: --- Comparação T1: Original vs Otimizado ---
# EN-US: --- T1 Comparison: Original vs. Optimized ---
plt.figure(figsize=(14, 7))
plt.plot(data_log_T1['t'], data_log_T1['xb'] * 1000, label='Posição $x_b(t)$ (T1 Original)', color='blue', linestyle=':')
plt.plot(data_log_T1_opt['t'], data_log_T1_opt['xb'] * 1000, label='Posição $x_b(t)$ (T1 Otimizado GA)', color='deepskyblue', linestyle='-')
plt.plot(data_log_T1['t'], data_log_T1['r_xb'] * 1000, label='Referência $r_{x_b}(t)$', linestyle=':', color='k')
plt.ylabel('Posição da Esfera (mm)')
plt.xlabel('Tempo (s)')
plt.title('Comparação T1 Original vs. T1 Otimizado (GA nos Polos)')
plt.legend()
plt.grid(True)
plt.ylim(0, maglev.x_b_max * 1000 + 1)
plt.show()

# PT-BR: --- Comparação T2I: Original vs Otimizado ---
# EN-US: --- T2I Comparison: Original vs. Optimized ---
plt.figure(figsize=(14, 7))
plt.plot(data_log_T2I['t'], data_log_T2I['xb'] * 1000, label='Posição $x_b(t)$ (T2I Original)', color='green', linestyle=':')
plt.plot(data_log_T2I_opt['t'], data_log_T2I_opt['xb'] * 1000, label='Posição $x_b(t)$ (T2I Otimizado GA)', color='limegreen', linestyle='-')
plt.plot(data_log_T2I['t'], data_log_T2I['r_xb'] * 1000, label='Referência $r_{x_b}(t)$', linestyle=':', color='k')
plt.ylabel('Posição da Esfera (mm)')
plt.xlabel('Tempo (s)')
plt.title('Comparação T2I Original vs. T2I Otimizado (GA nos Polos)')
plt.legend()
plt.grid(True)
plt.ylim(0, maglev.x_b_max * 1000 + 1)
plt.show()

# PT-BR: --- Plotagem Geral Comparativa: Todos os 4 + Referência ---
# EN-US: --- Overall Comparative Plot: All Four Controllers + Reference ---
plt.figure(figsize=(16, 8))
plt.plot(data_log_T1['t'], data_log_T1['xb'] * 1000, label='T1 Original', color='blue', linestyle=':')
plt.plot(data_log_T1_opt['t'], data_log_T1_opt['xb'] * 1000, label='T1 Otimizado GA', color='deepskyblue', linestyle='-')
plt.plot(data_log_T2I['t'], data_log_T2I['xb'] * 1000, label='T2I Original', color='green', linestyle=':')
plt.plot(data_log_T2I_opt['t'], data_log_T2I_opt['xb'] * 1000, label='T2I Otimizado GA', color='limegreen', linestyle='-')
plt.plot(data_log_T1['t'], data_log_T1['r_xb'] * 1000, label='Referência', linestyle='--', color='k', alpha=0.7)
plt.ylabel('Posição da Esfera (mm)')
plt.xlabel('Tempo (s)')
plt.title('Comparação Geral: Controladores Originais vs. Otimizados (GA nos Polos)')
plt.legend(loc='best')
plt.grid(True)
plt.ylim(0, maglev.x_b_max * 1000 + 1)
plt.show()

# PT-BR: --- Plotagem das Tensões de Controle ---
# EN-US: --- Control-Voltage Plot ---
plt.figure(figsize=(16, 8))
plt.plot(data_log_T1['t'], data_log_T1['Vc'], label='Vc T1 Original', color='blue', linestyle=':', alpha=0.7)
plt.plot(data_log_T1_opt['t'], data_log_T1_opt['Vc'], label='Vc T1 Otimizado GA', color='deepskyblue', linestyle='-', alpha=0.7)
plt.plot(data_log_T2I['t'], data_log_T2I['Vc'], label='Vc T2I Original', color='green', linestyle=':', alpha=0.7)
plt.plot(data_log_T2I_opt['t'], data_log_T2I_opt['Vc'], label='Vc T2I Otimizado GA', color='limegreen', linestyle='-', alpha=0.7)
plt.ylabel('Tensão de Controle (V)')
plt.xlabel('Tempo (s)')
plt.title('Comparação Geral: Tensões de Controle')
plt.legend(loc='best')
plt.grid(True)
plt.ylim(-maglev.V_max - 1, maglev.V_max + 1)
plt.show()


In [ ]:
# =============================================================================
# PT-BR: CÁLCULO DAS MÉTRICAS DE DESEMPENHO - TODOS OS CONTROLADORES
# EN-US: PERFORMANCE-METRIC CALCULATION — ALL CONTROLLERS
# PT-BR: Original (T1/T2I) + GA Standard + QGA (10, 16, 24, 32 bits)
# EN-US: Original (T1/T2I) + Standard GA + QGA (10, 16, 24, 32 bits)
# =============================================================================

def _calcular_metricas_log(nome_ctrl, data_log):
    """PT-BR:
    Calcula IAE, ITAE, ITSE e IG para um dado log de simulação.
    Retorna dicionário com os valores ou None se dados insuficientes.

    EN-US:
    Computes IAE, ITAE, ITSE, and IG for a simulation log.
    Returns a dictionary of values, or None when the available data are insufficient.
    """
    print(f"\n--- Métricas de Desempenho: {nome_ctrl} ---")
    if data_log is None:
        print("  AVISO: Log não disponível.")
        return None

    error          = data_log['r_xb'] - data_log['xb']
    control_signal = data_log['Vc']
    time_sig       = data_log['t']

    valid = ~np.isnan(error) & ~np.isnan(control_signal) & ~np.isnan(time_sig)
    error_c   = error[valid]
    control_c = control_signal[valid]
    time_c    = time_sig[valid]

    if len(error_c) <= 1:
        print("  ERRO: Dados insuficientes após remoção de NaNs.")
        return None

    dt = time_c[1] - time_c[0] if len(time_c) > 1 else sim_dt_controller

    iae_val  = calculate_iae(error_c, dt)
    itae_val = calculate_itae(error_c, time_c, dt)
    itse_val = calculate_itse(error_c, time_c, dt)
    ig_val   = calculate_ig(error_c, control_c, time_c, 0.2, 0.2, 0.6, dt)

    print(f"  IAE  : {iae_val:.4e}")
    print(f"  ITAE : {itae_val:.4e}")
    print(f"  ITSE : {itse_val:.4e}")
    print(f"  IG (α1=0.2, α2=0.2, α3=0.6): {ig_val:.4e}")

    return {'IAE': iae_val, 'ITAE': itae_val, 'ITSE': itse_val, 'IG': ig_val}


# =============================================================================
# PT-BR: Dicionário global de métricas - indexado por rótulo do controlador
# EN-US: Global metric dictionary indexed by controller label
# =============================================================================
todas_metricas = {}

# PT-BR: --- 1. Originais ---
# EN-US: --- 1. Original controllers ---
todas_metricas['T1 Original']  = _calcular_metricas_log('Controlador T1 Original',  data_log_T1)
todas_metricas['T2I Original'] = _calcular_metricas_log('Controlador T2I Original', data_log_T2I)

# PT-BR: --- 2. GA Standard ---
# EN-US: --- 2. Standard GA ---
todas_metricas['T1 GA']  = _calcular_metricas_log('T1 Otimizado (GA Standard)',  data_log_T1_opt)
todas_metricas['T2I GA'] = _calcular_metricas_log('T2I Otimizado (GA Standard)', data_log_T2I_opt)

# PT-BR: --- 3. QGA por bits ---
# EN-US: --- 3. QGA by bit resolution ---
for bits in bits_list:
    log_t1  = data_logs_qga_T1 .get(bits)
    log_t2i = data_logs_qga_T2I.get(bits)
    todas_metricas[f'T1 QGA {bits}b']  = _calcular_metricas_log(f'T1 QGA {bits} bits',  log_t1)
    todas_metricas[f'T2I QGA {bits}b'] = _calcular_metricas_log(f'T2I QGA {bits} bits', log_t2i)

# =============================================================================
# PT-BR: Tabela Resumo - todos os controladores
# EN-US: Summary Table — all controllers
# =============================================================================
print("\n" + "="*80)
print("TABELA RESUMO - MÉTRICAS DE DESEMPENHO (todos os controladores)")
print("="*80)
print(f"{'Controlador':<20} {'IAE':>12} {'ITAE':>12} {'ITSE':>12} {'IG':>12}")
print("-"*80)

for nome, m in todas_metricas.items():
    if m is not None:
        print(f"{nome:<20} {m['IAE']:>12.4e} {m['ITAE']:>12.4e} "
              f"{m['ITSE']:>12.4e} {m['IG']:>12.4e}")
    else:
        print(f"{nome:<20} {'N/A':>12} {'N/A':>12} {'N/A':>12} {'N/A':>12}")

print("="*80)

# =============================================================================
# PT-BR: Ranking por ITAE
# EN-US: Ranking by ITAE
# =============================================================================
print("\nRANKING POR ITAE (menor é melhor):")
ranking = sorted(
    [(n, m) for n, m in todas_metricas.items() if m is not None],
    key=lambda x: x[1]['ITAE']
)
melhor_itae = ranking[0][1]['ITAE'] if ranking else 1.0
for i, (nome, m) in enumerate(ranking, 1):
    delta = ((m['ITAE'] - melhor_itae) / melhor_itae) * 100
    sinal = f" (+{delta:.1f}%)" if i > 1 else " (melhor)"
    print(f"  {i:2}. {nome:<20} ITAE={m['ITAE']:.4e}{sinal}")

# =============================================================================
# PT-BR: Ranking por IG
# EN-US: Ranking by IG
# =============================================================================
print("\nRANKING POR ÍNDICE DE GOODHART (menor é melhor):")
ranking_ig = sorted(
    [(n, m) for n, m in todas_metricas.items() if m is not None],
    key=lambda x: x[1]['IG']
)
melhor_ig = ranking_ig[0][1]['IG'] if ranking_ig else 1.0
for i, (nome, m) in enumerate(ranking_ig, 1):
    delta = ((m['IG'] - melhor_ig) / melhor_ig) * 100
    sinal = f" (+{delta:.1f}%)" if i > 1 else " (melhor)"
    print(f"  {i:2}. {nome:<20} IG={m['IG']:.4e}{sinal}")

# PT-BR: A Tabela 5.1 da tese apresenta esses indicadores para o MAGLEV.
# EN-US: Dissertation Table 5.1 presents these indicators for the MAGLEV system.

In [ ]:
# =============================================================================
# PT-BR: ANÁLISE COMPARATIVA COMPLETA: Original + GA Standard + QGA (multi-bits)
# EN-US: COMPLETE COMPARATIVE ANALYSIS: Original + Standard GA + QGA (multiple bit resolutions)
# PT-BR: Depende das células anteriores: all_dados, bits_list, data_logs_qga_T1,
# EN-US: Depends on the preceding cells: all_dados, bits_list, data_logs_qga_T1,
# PT-BR: data_logs_qga_T2I, polos_qga_T1, polos_qga_T2I, todas_metricas
# EN-US: data_logs_qga_T2I, polos_qga_T1, polos_qga_T2I, todas_metricas
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt

# PT-BR: --- Função vetorizada de referência (compatibilidade com plotagem) ---
# EN-US: --- Vectorized Reference Function (plotting compatibility) ---
def reference_signal_sim_vectorized(t_array):
    t = np.asarray(t_array)
    return np.select(
        [t < 0.5, (t >= 0.5) & (t < 2.5), (t >= 2.5) & (t < 5.0), (t >= 5.0) & (t < 7.5)],
        [sim_initial_xb, 0.005, 0.009, 0.003],
        default=0.007
    )

# =============================================================================
# PT-BR: FUNÇÕES AUXILIARES DE MÉTRICAS
# EN-US: METRIC HELPER FUNCTIONS
# =============================================================================
def calculate_settling_time(output_signal, reference_signal, time_signal, tolerance_percent=2.0):
    """PT-BR:
    Tempo de acomodação relativo a cada degrau de referência.

    EN-US:
    Computes the settling time relative to each reference step.
    """
    try:
        change_indices = np.where(np.abs(np.diff(reference_signal)) > 0.0005)[0]
        if len(change_indices) == 0:
            return 0.0
        settling_times = []
        for change_idx in change_indices:
            if change_idx + 100 >= len(output_signal):
                continue
            ref_final    = reference_signal[change_idx + 10]
            tolerance_abs = (tolerance_percent / 100.0) * ref_final
            segment_start = change_idx + 50
            segment_end   = min(change_idx + 500, len(output_signal) - 1)
            for i in range(segment_end, segment_start, -1):
                if abs(output_signal[i] - ref_final) > tolerance_abs:
                    settling_times.append(time_signal[i] - time_signal[change_idx])
                    break
        return max(settling_times) if settling_times else 0.0
    except Exception as e:
        print(f"Erro no cálculo do tempo de acomodação: {e}")
        return np.nan

def calculate_overshoot(output_signal, reference_signal, time_signal, min_change_threshold=0.0003):
    """PT-BR:
    Overshoot/undershoot máximo em % para degraus na referência.

    EN-US:
    Computes the maximum overshoot/undershoot percentage for reference steps.
    """
    try:
        change_indices = np.where(np.abs(np.diff(reference_signal)) > min_change_threshold)[0]
        if len(change_indices) == 0:
            return 0.0
        max_overshoot = 0.0
        for change_idx in change_indices:
            if change_idx + 50 >= len(output_signal):
                continue
            ref_before = reference_signal[max(0, change_idx - 5)]
            ref_after  = reference_signal[min(len(reference_signal) - 1, change_idx + 20)]
            step_size  = abs(ref_after - ref_before)
            if step_size < min_change_threshold:
                continue
            analysis_end = min(change_idx + 500, len(output_signal))
            segment = output_signal[change_idx:analysis_end]
            if ref_after > ref_before:
                max_val = np.max(segment)
                if max_val > ref_after:
                    max_overshoot = max(max_overshoot, ((max_val - ref_after) / step_size) * 100)
            else:
                min_val = np.min(segment)
                if min_val < ref_after:
                    max_overshoot = max(max_overshoot, ((ref_after - min_val) / step_size) * 100)
        return max(0.0, max_overshoot)
    except Exception as e:
        print(f"Erro no cálculo do overshoot: {e}")
        return 0.0

def calculate_rms_error(error_signal):
    try:
        # PT-BR: mm
        # EN-US: mm
        return np.sqrt(np.mean(error_signal**2)) * 1000
    except:
        return np.nan

def display_metrics_complete(controller_name, data_log, polos_complexos=None, polos_cromossomo=None):
    """PT-BR:
    Calcula e exibe métricas completas. Retorna dicionário ou None.

    EN-US:
    Computes and displays the complete metric set. Returns a dictionary or None.
    """
    print(f"\n--- Métricas: {controller_name} ---")
    if polos_complexos is not None:
        print(f"  Polos: {polos_complexos}")
    if polos_cromossomo is not None:
        print(f"  Cromossomo: {np.round(polos_cromossomo, 4)}")

    if data_log is None or not all(k in data_log for k in ['r_xb', 'xb', 't', 'Vc']):
        print(f"  AVISO: Dados ausentes ou incompletos.")
        return None

    error   = data_log['r_xb'] - data_log['xb']
    control = data_log['Vc']
    time    = data_log['t']

    valid = ~np.isnan(error) & ~np.isnan(control) & ~np.isnan(time)
    ec, cc, tc = error[valid], control[valid], time[valid]

    if len(ec) < 2:
        print("  ERRO: Dados insuficientes após limpeza de NaN.")
        return None

    dt = tc[1] - tc[0] if len(tc) > 1 else 0.002

    iae  = calculate_iae(ec, dt)
    itae = calculate_itae(ec, tc, dt)
    itse = calculate_itse(ec, tc, dt)
    ig   = calculate_ig(ec, cc, tc, 0.2, 0.2, 0.6, dt)

    max_err     = np.max(np.abs(ec)) * 1000
    rms_err     = calculate_rms_error(ec)
    settle      = calculate_settling_time(data_log['xb'], data_log['r_xb'], data_log['t'])
    overshoot   = calculate_overshoot(data_log['xb'], data_log['r_xb'], data_log['t'])
    ctrl_effort = np.sum(np.abs(cc)) * dt

    print(f"  IAE={iae:.4e} | ITAE={itae:.4e} | ITSE={itse:.4e} | IG={ig:.4e}")
    print(f"  Erro Máx={max_err:.3f}mm | RMS={rms_err:.3f}mm | "
          f"T_Acom={settle:.3f}s | Overshoot={overshoot:.2f}% | Esforço={ctrl_effort:.3f}V·s")

    return {
        'IAE': iae, 'ITAE': itae, 'ITSE': itse, 'IG': ig,
        'Erro_Max_mm': max_err, 'Erro_RMS_mm': rms_err,
        'Tempo_Acomodacao_s': settle, 'Overshoot_%': overshoot,
        'Esforco_Controle_Vs': ctrl_effort,
        'Polos': str(polos_complexos) if polos_complexos is not None else 'N/A',
        'Cromossomo': str(np.round(polos_cromossomo, 4)) if polos_cromossomo is not None else 'N/A'
    }

# =============================================================================
# PT-BR: PLOTAGEM DETALHADA POR BITS (4 painéis: posição, erro, Vc, ic)
# EN-US: DETAILED PLOTS BY BIT RESOLUTION (4 panels: position, error, Vc, ic)
# =============================================================================
cores_qga = {10: 'crimson', 16: 'darkorange', 24: 'purple', 32: 'saddlebrown'}

for tipo_ctrl in ['T1', 'T2I']:
    logs_qga = data_logs_qga_T1 if tipo_ctrl == 'T1' else data_logs_qga_T2I
    for bits in bits_list:
        log = logs_qga.get(bits)
        if log is None:
            print(f"AVISO: Sem dados para {tipo_ctrl} QGA {bits}bits. Pulando subplot.")
            continue

        fig, axs = plt.subplots(4, 1, figsize=(12, 13), sharex=True)
        fig.suptitle(
            f'MAGLEV - Controlador Fuzzy {tipo_ctrl} Otimizado (QGA {bits} bits)',
            fontsize=14, fontweight='bold'
        )

        cor = cores_qga[bits]
        t   = log['t']
        ref = reference_signal_sim_vectorized(t)

        axs[0].plot(t, log['xb'] * 1000, label='$x_b(t)$', color=cor, linewidth=2)
        axs[0].plot(t, ref * 1000, label='$r(t)$', color='k', linestyle='--', linewidth=1.5)
        axs[0].set_ylabel('Posição (mm)')
        axs[0].legend()
        axs[0].grid(True)
        if 'maglev' in globals() and maglev:
            axs[0].set_ylim(0, maglev.x_b_max * 1000 + 1)

        axs[1].plot(t, (ref - log['xb']) * 1000, label='$e(t)$',
                    color='orange' if tipo_ctrl == 'T1' else 'purple', linewidth=1.5)
        axs[1].set_ylabel('Erro (mm)')
        axs[1].legend()
        axs[1].grid(True)

        axs[2].plot(t, log['Vc'], label='$V_c(t)$', color='r', linewidth=1.5)
        axs[2].set_ylabel('Tensão (V)')
        axs[2].legend()
        axs[2].grid(True)
        if 'maglev' in globals() and maglev:
            axs[2].set_ylim(-maglev.V_max - 1, maglev.V_max + 1)

        axs[3].plot(t, log['ic'], label='$i_c(t)$', color='g', linewidth=1.5)
        axs[3].set_xlabel('Tempo (s)')
        axs[3].set_ylabel('Corrente (A)')
        axs[3].legend()
        axs[3].grid(True)

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

# =============================================================================
# PT-BR: PLOTAGEM COMPARATIVA: GA Standard vs. QGA (todos os bits) - Posição
# EN-US: COMPARATIVE PLOT: Standard GA vs. QGA (all bit resolutions) — Position
# =============================================================================
for tipo_ctrl in ['T1', 'T2I']:
    fig, ax = plt.subplots(figsize=(16, 7))
    logs_qga = data_logs_qga_T1 if tipo_ctrl == 'T1' else data_logs_qga_T2I
    log_ga   = data_log_T1_opt  if tipo_ctrl == 'T1' else data_log_T2I_opt
    ref_time = None

    if log_ga is not None and 'xb' in log_ga:
        ax.plot(log_ga['t'], log_ga['xb'] * 1000,
                label=f'{tipo_ctrl} GA Standard',
                color='deepskyblue' if tipo_ctrl == 'T1' else 'limegreen',
                linestyle='--', linewidth=2)
        ref_time = log_ga['t']

    ls_map = {10: '-', 16: '-.', 24: (0,(3,1,1,1)), 32: ':'}
    for bits in bits_list:
        log = logs_qga.get(bits)
        if log is not None and 'xb' in log:
            ax.plot(log['t'], log['xb'] * 1000,
                    label=f'{tipo_ctrl} QGA {bits}bits',
                    color=cores_qga[bits], linestyle=ls_map[bits], linewidth=2)
            if ref_time is None:
                ref_time = log['t']

    if ref_time is not None:
        ax.plot(ref_time, reference_signal_sim_vectorized(ref_time) * 1000,
                label='Referência $r(t)$', color='k', linestyle=':', linewidth=2, alpha=0.8)

    ax.set_ylabel('Posição da Esfera (mm)')
    ax.set_xlabel('Tempo (s)')
    ax.set_title(f'Comparação de Posição - {tipo_ctrl}: GA Standard vs. QGA (multi-bits)')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True)
    if 'maglev' in globals() and maglev:
        ax.set_ylim(0, maglev.x_b_max * 1000 + 1)
    plt.tight_layout()
    plt.show()

# =============================================================================
# PT-BR: PLOTAGEM COMPARATIVA: Tensão de Controle
# EN-US: COMPARATIVE PLOT: Control Voltage
# =============================================================================
for tipo_ctrl in ['T1', 'T2I']:
    fig, ax = plt.subplots(figsize=(16, 5))
    logs_qga = data_logs_qga_T1 if tipo_ctrl == 'T1' else data_logs_qga_T2I
    log_ga   = data_log_T1_opt  if tipo_ctrl == 'T1' else data_log_T2I_opt

    if log_ga is not None and 'Vc' in log_ga:
        ax.plot(log_ga['t'], log_ga['Vc'],
                label=f'{tipo_ctrl} GA Standard',
                color='deepskyblue' if tipo_ctrl == 'T1' else 'limegreen',
                linestyle='--', linewidth=2, alpha=0.85)

    ls_map = {10: '-', 16: '-.', 24: (0,(3,1,1,1)), 32: ':'}
    for bits in bits_list:
        log = logs_qga.get(bits)
        if log is not None and 'Vc' in log:
            ax.plot(log['t'], log['Vc'],
                    label=f'{tipo_ctrl} QGA {bits}bits',
                    color=cores_qga[bits], linestyle=ls_map[bits], linewidth=2, alpha=0.85)

    ax.set_ylabel('Tensão de Controle (V)')
    ax.set_xlabel('Tempo (s)')
    ax.set_title(f'Comparação de Tensão - {tipo_ctrl}: GA Standard vs. QGA (multi-bits)')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True)
    if 'maglev' in globals() and maglev:
        ax.set_ylim(-maglev.V_max - 1, maglev.V_max + 1)
    plt.tight_layout()
    plt.show()

# =============================================================================
# PT-BR: CÁLCULO DE MÉTRICAS - todos os controladores + todos os bits QGA
# EN-US: METRIC CALCULATION — all controllers + all QGA bit resolutions
# =============================================================================
print("\n" + "="*80)
print("ANÁLISE COMPARATIVA DETALHADA - TODOS OS CONTROLADORES E BITS")
print("="*80)

performance_summary = {}

# PT-BR: --- Originais ---
# EN-US: --- Original controllers ---
for nome, var in [('T1 Original', 'data_log_T1'), ('T2I Original', 'data_log_T2I')]:
    log = globals().get(var)
    m = display_metrics_complete(nome, log)
    if m:
        performance_summary[nome] = m

# PT-BR: --- GA Standard ---
# EN-US: --- Standard GA ---
for nome, var, polo_var in [
    ('T1 GA Standard',  'data_log_T1_opt',  'optimized_desired_poles_T1'),
    ('T2I GA Standard', 'data_log_T2I_opt',  'optimized_desired_poles_T2I'),
]:
    log  = globals().get(var)
    polo = globals().get(polo_var)
    m = display_metrics_complete(nome, log, polos_complexos=polo)
    if m:
        performance_summary[nome] = m

# PT-BR: --- QGA por bits ---
# EN-US: --- QGA by bit resolution ---
for bits in bits_list:
    for tipo_ctrl in ['T1', 'T2I']:
        nome = f'{tipo_ctrl} QGA {bits}bits'
        logs_qga = data_logs_qga_T1 if tipo_ctrl == 'T1' else data_logs_qga_T2I
        polos_qga_dict = polos_qga_T1 if tipo_ctrl == 'T1' else polos_qga_T2I
        log  = logs_qga.get(bits)
        polo = polos_qga_dict.get(bits)
        m = display_metrics_complete(nome, log, polos_complexos=polo)
        if m:
            performance_summary[nome] = m

# =============================================================================
# PT-BR: GRÁFICO DE BARRAS COMPARATIVO (6 métricas)
# EN-US: COMPARATIVE BAR CHART (6 metrics)
# =============================================================================
if performance_summary and len(performance_summary) > 1:
    try:
        fig, axes = plt.subplots(3, 2, figsize=(18, 13))
        fig.suptitle('Comparação de Métricas de Desempenho - Todos os Controladores', fontsize=16)

        controllers = list(performance_summary.keys())
        metricas_plot = [
            ('ITAE',               'ITAE',               'steelblue',   axes[0,0]),
            ('IG',                 'Índice de Goodhart',  'gold',        axes[0,1]),
            ('Erro_RMS_mm',        'Erro RMS (mm)',       'coral',       axes[1,0]),
            ('Esforco_Controle_Vs','Esforço Controle (V·s)', 'lightgreen', axes[1,1]),
            ('Tempo_Acomodacao_s', 'Tempo de Acomodação (s)', 'skyblue', axes[2,0]),
            ('Overshoot_%',        'Overshoot (%)',       'lightcoral',  axes[2,1]),
        ]

        for chave, titulo, cor, ax in metricas_plot:
            vals = [performance_summary[c][chave] for c in controllers]
            ax.bar(controllers, vals, color=cor, alpha=0.8)
            ax.set_title(titulo, fontweight='bold')
            ax.set_ylabel(titulo)
            for lbl in ax.get_xticklabels():
                lbl.set_rotation(45)
                lbl.set_ha('right')
            ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Erro ao criar gráficos de barras: {e}")
        import traceback; traceback.print_exc()

# =============================================================================
# PT-BR: RANKING E TABELA RESUMO
# EN-US: RANKING AND SUMMARY TABLE
# =============================================================================
sorted_by_itae = sorted(
    [(n, m) for n, m in performance_summary.items() if m is not None],
    key=lambda x: x[1]['ITAE']
)
sorted_by_ig = sorted(
    [(n, m) for n, m in performance_summary.items() if m is not None],
    key=lambda x: x[1]['IG']
)

print("\n" + "-"*70)
print("RANKING POR ITAE (menor é melhor):")
print("-"*70)
best_itae = sorted_by_itae[0][1]['ITAE']
for i, (nome, m) in enumerate(sorted_by_itae, 1):
    delta = f" (+{((m['ITAE']-best_itae)/best_itae)*100:.1f}%)" if i > 1 else " (melhor)"
    print(f"  {i:2}. {nome:<22} ITAE={m['ITAE']:.4e}{delta} | "
          f"IG={m['IG']:.4e} | T_Acom={m['Tempo_Acomodacao_s']:.3f}s | "
          f"Ovr={m['Overshoot_%']:.2f}%")

print("\n" + "-"*70)
print("RANKING POR IG (menor é melhor):")
print("-"*70)
best_ig = sorted_by_ig[0][1]['IG']
for i, (nome, m) in enumerate(sorted_by_ig, 1):
    delta = f" (+{((m['IG']-best_ig)/best_ig)*100:.1f}%)" if i > 1 else " (melhor)"
    print(f"  {i:2}. {nome:<22} IG={m['IG']:.4e}{delta} | ITAE={m['ITAE']:.4e}")

print("\n" + "="*100)
print("TABELA RESUMO - TODAS AS MÉTRICAS")
print("="*100)
print(f"{'Controlador':<22} {'ITAE':>12} {'IG':>12} {'T.Acom(s)':>10} "
      f"{'Ovr(%)':>8} {'RMS(mm)':>9} {'Esf(V·s)':>9}")
print("-"*100)
for nome, m in sorted_by_itae:
    print(f"{nome:<22} {m['ITAE']:>12.4e} {m['IG']:>12.4e} "
          f"{m['Tempo_Acomodacao_s']:>10.3f} {m['Overshoot_%']:>8.2f} "
          f"{m['Erro_RMS_mm']:>9.3f} {m['Esforco_Controle_Vs']:>9.3f}")

print("\n" + "="*80)
print("FIM DA ANÁLISE COMPARATIVA COMPLETA")
print("="*80)

In [ ]:
# PT-BR: Configuração de estilo para artigo
# EN-US: Publication-style configuration
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'legend.fontsize': 12,
    'lines.linewidth': 2,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.format': 'pdf',
    'savefig.bbox': 'tight'
})

# PT-BR: --- Figura 1: MFs T1 para Posição ---
# EN-US: --- Figure 1: T1 MFs for Position ---
fig, ax = plt.subplots(figsize=(12, 5))

x_pos = np.linspace(0.001, 0.013, 1000)
colors = plt.cm.tab10(np.linspace(0, 1, len(mfs_xb_t1_names)))

for idx, mf_name in enumerate(mfs_xb_t1_names):
    mf_vals = [calculate_mf_value(x, mf_name, mfs_xb_t1_params) for x in x_pos]
    ax.plot(x_pos * 1000, mf_vals, label=mf_name, color=colors[idx], linewidth=2)

ax.fill_between(x_pos * 1000, 0, 1, alpha=0.05, color='gray')
ax.set_xlabel('Posição da Esfera $x_b(t)$ (mm)')
ax.set_ylabel('Grau de Pertinência')
ax.set_title('Funções de Pertinência Fuzzy Tipo-1 para Posição da Esfera', fontweight='bold')
ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.15), frameon=True)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 14)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mf_t1_position.pdf', dpi=300, bbox_inches='tight')
plt.show()

# PT-BR: --- Figura 2: MFs T1 para Erro ---
# EN-US: --- Figure 2: T1 MFs for Error ---
fig, ax = plt.subplots(figsize=(12, 5))

# PT-BR: Range aumentado para incluir Nmax e Pmax (que vão até 2*err_domain_half_width)
# EN-US: Range expanded to include Nmax and Pmax (which extend to 2*err_domain_half_width)
err_range = np.linspace(-2.0*err_domain_half_width, 2.0*err_domain_half_width, 1000)
colors_err = plt.cm.tab20(np.linspace(0, 1, len(mfs_v_dot_xb_t1_names)))

for idx, mf_name in enumerate(mfs_v_dot_xb_t1_names):
    mf_vals = [calculate_mf_value(e, mf_name, mfs_v_dot_xb_t1_params) for e in err_range]
    ax.plot(err_range * 1000, mf_vals, label=mf_name, color=colors_err[idx], linewidth=2)

ax.axvline(0, color='k', linestyle=':', alpha=0.5, linewidth=1.5)
ax.set_xlabel('Erro de Posição $e(t) = r(t) - x_b(t)$ (mm)')
ax.set_ylabel('Grau de Pertinência')
ax.set_title('Funções de Pertinência Fuzzy Tipo-1 para Erro de Posição', fontweight='bold')
ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.15), frameon=True)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mf_t1_error.pdf', dpi=300, bbox_inches='tight')
plt.show()

# PT-BR: --- Figura 3a: MFs T2I com FOU - Posição ---
# EN-US: --- Figure 3a: T2I MFs with FOU — Position ---
fig, ax = plt.subplots(figsize=(12, 6))

x_pos_t2i = np.linspace(0.001, 0.013, 1000)
colors_t2i = plt.cm.tab10(np.linspace(0, 1, len(mfs_xb_t2i_names_actual)))

for idx, mf_name in enumerate(mfs_xb_t2i_names_actual):
    umf_vals = np.array([calculate_mf_value(x, mf_name, mfs_xb_t2i_params_umf_actual) for x in x_pos_t2i])
    lmf_vals = np.array([calculate_mf_value(x, mf_name, mfs_xb_t2i_params_lmf_actual) for x in x_pos_t2i])
    
    # PT-BR: Limita os valores ao intervalo válido [0, 1]
    # EN-US: Clip values to the valid range [0, 1]
    umf_vals = np.clip(umf_vals, 0, 1)
    lmf_vals = np.clip(lmf_vals, 0, 1)
    
    ax.plot(x_pos_t2i * 1000, umf_vals, linewidth=2.5, color=colors_t2i[idx], label=f'{mf_name} (UMF)', linestyle='-')
    ax.plot(x_pos_t2i * 1000, lmf_vals, linewidth=2, color=colors_t2i[idx], linestyle='--', alpha=0.7)
    ax.fill_between(x_pos_t2i * 1000, lmf_vals, umf_vals, alpha=0.2, color=colors_t2i[idx])

ax.set_xlabel('Posição da Esfera $x_b(t)$ (mm)', fontsize=14)
ax.set_ylabel('Grau de Pertinência', fontsize=14)
ax.set_title('Funções de Pertinência Fuzzy Tipo-2 Intervalar para Posição da Esfera', fontweight='bold', fontsize=16)
ax.grid(True, alpha=0.3)
# PT-BR: Ajustado para corresponder ao intervalo dos dados
# EN-US: Adjusted to match the data range
ax.set_xlim(1, 13)
ax.set_ylim(0, 1.1)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=True, fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mf_t2i_position_fou.pdf', dpi=300, bbox_inches='tight')
plt.show()

# PT-BR: --- Figura 3b: MFs T2I com FOU - Erro ---
# EN-US: --- Figure 3b: T2I MFs with FOU — Error ---
fig, ax = plt.subplots(figsize=(12, 6))

# PT-BR: Aumentar o range para mostrar Pmax e Nmax corretamente
# EN-US: Expands the range to display Pmax and Nmax correctly
err_range_t2i = np.linspace(-2.0 * err_lim_t2i_fig, 2.0 * err_lim_t2i_fig, 1000)

# PT-BR: Usa as chaves do dicionário de parâmetros para garantir que todas as MFs (incluindo Pmax/Nmax) sejam plotadas
# EN-US: Uses the parameter-dictionary keys to ensure that every MF (including Pmax/Nmax) is plotted
mf_names_t2i_error = list(mfs_v_dot_xb_t2i_params_umf_actual.keys())
colors_err_t2i = plt.cm.tab20(np.linspace(0, 1, len(mf_names_t2i_error)))

for idx, mf_name in enumerate(mf_names_t2i_error):
    umf_vals = [calculate_mf_value(e, mf_name, mfs_v_dot_xb_t2i_params_umf_actual) for e in err_range_t2i]
    lmf_vals = [calculate_mf_value(e, mf_name, mfs_v_dot_xb_t2i_params_lmf_actual) for e in err_range_t2i]
    
    ax.plot(err_range_t2i * 1000, umf_vals, linewidth=2.5, color=colors_err_t2i[idx], label=f'{mf_name} (UMF)', linestyle='-')
    ax.plot(err_range_t2i * 1000, lmf_vals, linewidth=2, color=colors_err_t2i[idx], linestyle='--', alpha=0.7)
    ax.fill_between(err_range_t2i * 1000, lmf_vals, umf_vals, alpha=0.2, color=colors_err_t2i[idx])

ax.axvline(0, color='k', linestyle=':', alpha=0.5, linewidth=1.5)
ax.set_xlabel('Erro de Posição $e(t) = r(t) - x_b(t)$ (mm)', fontsize=14)
ax.set_ylabel('Grau de Pertinência', fontsize=14)
ax.set_title('Funções de Pertinência Fuzzy Tipo-2 Intervalar para Erro de Posição', fontweight='bold', fontsize=16)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.1)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=True, fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mf_t2i_error_fou.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# PT-BR: Curvas e gráficos de convergência
# EN-US: Convergence curves and plots

if not DATA_LOADED_SUCCESSFULLY:
    print("ERRO: Dados não carregados. Execute o bloco de carregamento primeiro.")
else:
    # -----------------------------------------------------------------------
    # PT-BR: Funções auxiliares
    # EN-US: Helper functions
    # -----------------------------------------------------------------------
    def extrair_itae_ga(historico):
        """PT-BR:
        Extrai ITAE por geração do histórico do GA Standard (lista de floats).

        EN-US:
        Extracts ITAE by generation from the Standard GA history (a list of floats).
        """
        return [float(v) for v in historico]

    def extrair_itae_qga(historico):
        """PT-BR:
        Extrai ITAE por geração do histórico do QGA (lista de dicts ou floats).

        EN-US:
        Extracts ITAE by generation from the QGA history (a list of dictionaries or floats).
        """
        itae_list = []
        for h in historico:
            if isinstance(h, dict):
                apt = h.get('melhor_aptidao_global', h.get('melhor_aptidao_geracao', 1e-9))
                itae = (1.0 / apt - 1e-9) if apt > 1e-9 else 1e9
            else:
                itae = float(h)
            itae_list.append(itae)
        return itae_list

    def geracao_convergencia(itae_serie, melhor_final, threshold_pct=1.0):
        """PT-BR:
        Retorna a primeira geração em que o algoritmo chegou a
        'threshold_pct'% do melhor valor final.

        EN-US:
        Returns the first generation in which the algorithm reaches a value within
        'threshold_pct'% of the final best value.
        """
        alvo = melhor_final * (1 + threshold_pct / 100.0)
        for i, v in enumerate(itae_serie):
            if v <= alvo:
                return i
        return len(itae_serie) - 1

    # -----------------------------------------------------------------------
    # PT-BR: GA Standard - histórico e tempos (referência de bits[0])
    # EN-US: Standard GA — history and run times (reference from bits[0])
    # -----------------------------------------------------------------------
    historico_ga_t1  = ga_historico_ga_t1  if 'ga_historico_ga_t1'  in globals() else None
    historico_ga_t2i = ga_historico_ga_t2i if 'ga_historico_ga_t2i' in globals() else None

    _t_ga_t1  = ga_tempo_ga_t1_s  if 'ga_tempo_ga_t1_s'  in globals() else 0.0
    _t_ga_t2i = ga_tempo_ga_t2i_s if 'ga_tempo_ga_t2i_s' in globals() else 0.0

    if historico_ga_t1 is None or historico_ga_t2i is None:
        print("AVISO: Histórico do GA Standard não encontrado.")
        print([k for k in globals() if k.startswith('ga_') or k.startswith('qga_')])
    else:
        itae_ga_t1  = extrair_itae_ga(historico_ga_t1[1:])
        itae_ga_t2i = extrair_itae_ga(historico_ga_t2i[1:])
        best_ga_t1  = min(itae_ga_t1)
        best_ga_t2i = min(itae_ga_t2i)

        # -----------------------------------------------------------------------
        # PT-BR: QGA - carrega histórico e tempos para cada resolução de bits
        # EN-US: QGA — loads the history and run times for each bit resolution
        # -----------------------------------------------------------------------
        # PT-BR: Cores e estilos de linha para cada bits
        # EN-US: Colors and line styles for each bit resolution
        cores_bits = {10: 'crimson', 16: 'darkorange', 24: 'purple', 32: 'saddlebrown'}
        ls_bits    = {10: '-',       16: '-.',          24: (0,(3,1,1,1)), 32: ':'}

        # PT-BR: historicos_qga_t1[bits]
        # EN-US: Structure of the QGA histories for the T1 controller by resolution
        historicos_qga_t1  = {}
        # PT-BR: historicos_qga_t2i[bits]
        # EN-US: Structure of the QGA histories for the T2I controller by resolution
        historicos_qga_t2i = {}
        tempos_qga_t1      = {}
        tempos_qga_t2i     = {}
        itaes_qga_t1       = {}
        itaes_qga_t2i      = {}
        bests_qga_t1       = {}
        bests_qga_t2i      = {}

        for bits in bits_list:
            dados_bits = all_dados.get(bits) or {}

            hist_t1  = dados_bits.get('qga_historico_qga_t1')
            hist_t2i = dados_bits.get('qga_historico_qga_t2i')
            t_t1     = dados_bits.get('qga_tempo_qga_t1_s',  0.0)
            t_t2i    = dados_bits.get('qga_tempo_qga_t2i_s', 0.0)

            historicos_qga_t1[bits]  = hist_t1
            historicos_qga_t2i[bits] = hist_t2i
            tempos_qga_t1[bits]      = t_t1
            tempos_qga_t2i[bits]     = t_t2i

            if hist_t1 is not None:
                itaes_qga_t1[bits]  = extrair_itae_qga(hist_t1)
                bests_qga_t1[bits]  = min(itaes_qga_t1[bits])
            if hist_t2i is not None:
                itaes_qga_t2i[bits] = extrair_itae_qga(hist_t2i)
                bests_qga_t2i[bits] = min(itaes_qga_t2i[bits])

        # -----------------------------------------------------------------------
        # PT-BR: Thresholds analisados
        # EN-US: Analyzed thresholds
        # -----------------------------------------------------------------------
        thresholds = [5.0, 2.0, 1.0]

        # -----------------------------------------------------------------------
        # PT-BR: GRÁFICOS DE CONVERGÊNCIA - 1 figura por controlador (T1 e T2I)
        # EN-US: CONVERGENCE PLOTS — one figure per controller (T1 and T2I)
        # -----------------------------------------------------------------------
        cores_threshold = {10.0: '#e41a1c', 5.0: '#ff7f00',
                           2.0: '#4daf4a', 1.0: '#984ea3', 0.5: '#377eb8'}

        for tipo_ctrl, itae_ga_serie, best_ga, itaes_qga_dict, tempos_qga_dict in [
            ('T1',  itae_ga_t1,  best_ga_t1,  itaes_qga_t1,  tempos_qga_t1),
            ('T2I', itae_ga_t2i, best_ga_t2i, itaes_qga_t2i, tempos_qga_t2i),
        ]:
            n_bits_ok = sum(1 for b in bits_list if itaes_qga_dict.get(b) is not None)
            # PT-BR: GA Standard + 1 por bits
            # EN-US: Standard GA + one panel per bit resolution
            n_panels  = n_bits_ok + 1

            # PT-BR: --- CORREÇÃO: calcula n_cols e n_rows a partir de n_panels ---
            # EN-US: --- CORRECTION: computes n_cols and n_rows from n_panels ---
            n_cols = 2 if n_panels > 1 else 1
            # PT-BR: ceil(n_panels / n_cols)
            # EN-US: ceil(n_panels / n_cols)
            n_rows = (n_panels + n_cols - 1) // n_cols

            fig, axes = plt.subplots(n_rows, n_cols,
                                     figsize=(8 * n_cols, 5.5 * n_rows),
                                     squeeze=False)
            axes_flat = axes.flatten()

            titulo_fig = (f'Convergência por Geração - Controlador {tipo_ctrl}\n'
                          f'GA Standard vs. QGA (10, 16, 24, 32 bits)')
            fig.suptitle(titulo_fig, fontsize=15, fontweight='bold')

            # PT-BR: --- Painel GA Standard ---
            # EN-US: --- Standard GA Panel ---
            ax_ga = axes_flat[0]
            ax_ga.plot(np.arange(len(itae_ga_serie)), itae_ga_serie,
                       color='steelblue', linewidth=2.5, label='GA Standard')

            for thr, cor_thr in cores_threshold.items():
                gen_thr = geracao_convergencia(itae_ga_serie, best_ga, thr)
                ax_ga.axvline(gen_thr, color=cor_thr, linestyle='--',
                              alpha=0.8, linewidth=1.5,
                              label=f'{thr:.1f}%-conv: gen {gen_thr}')

            ax_ga.set_title('GA Standard', fontweight='bold', fontsize=13)
            ax_ga.set_xlabel('Geração')
            ax_ga.set_ylabel('Melhor ITAE')
            ax_ga.set_yscale('log')
            ax_ga.legend(fontsize=9)
            ax_ga.grid(True, alpha=0.3)

            # PT-BR: --- Painéis QGA por bits ---
            # EN-US: --- QGA Panels by Bit Resolution ---
            panel_idx = 1
            for bits in bits_list:
                serie = itaes_qga_dict.get(bits)
                if serie is None:
                    continue
                best_qga = min(serie)
                ax_b = axes_flat[panel_idx]

                ax_b.plot(np.arange(len(serie)), serie,
                          color=cores_bits[bits], linewidth=2.5,
                          linestyle=ls_bits[bits],
                          label=f'QGA {bits}bits')

                ax_b.axhline(best_ga, color='steelblue', linestyle=':',
                             linewidth=1.5, alpha=0.8,
                             label=f'Melhor GA ({best_ga:.3e})')

                for thr, cor_thr in cores_threshold.items():
                    gen_thr = geracao_convergencia(serie, best_qga, thr)
                    ax_b.axvline(gen_thr, color=cor_thr, linestyle='--',
                                 alpha=0.8, linewidth=1.5,
                                 label=f'{thr:.1f}%-conv: gen {gen_thr}')

                ax_b.set_title(f'QGA {bits} bits', fontweight='bold', fontsize=13)
                ax_b.set_xlabel('Geração')
                ax_b.set_ylabel('Melhor ITAE')
                ax_b.set_yscale('log')
                ax_b.legend(fontsize=9)
                ax_b.grid(True, alpha=0.3)
                panel_idx += 1

            # PT-BR: Oculta painéis sobressalentes
            # EN-US: Hides unused panels
            for k in range(panel_idx, len(axes_flat)):
                axes_flat[k].set_visible(False)

            plt.tight_layout(rect=[0, 0, 1, 0.94])
            fname = f'convergencia_ga_vs_qga_{tipo_ctrl}_bits.pdf'
            plt.savefig(FIGURES_DIR / fname, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"Gráfico salvo em '{fname}'")
        # -----------------------------------------------------------------------
        # PT-BR: GRÁFICO OVERLAY - todos os bits no mesmo eixo (T1 e T2I)
        # EN-US: OVERLAY PLOT — all bit resolutions on the same axis (T1 and T2I)
        # -----------------------------------------------------------------------
        fig, axes_ov = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle('Convergência Comparativa: GA Standard vs. QGA (multi-bits)',
                     fontsize=15, fontweight='bold')

        for ax_ov, tipo_ctrl, itae_ga_serie, best_ga, itaes_qga_dict in [
            (axes_ov[0], 'T1',  itae_ga_t1,  best_ga_t1,  itaes_qga_t1),
            (axes_ov[1], 'T2I', itae_ga_t2i, best_ga_t2i, itaes_qga_t2i),
        ]:
            ax_ov.plot(np.arange(len(itae_ga_serie)), itae_ga_serie,
                       color='steelblue', linewidth=2.5, label='GA Standard', zorder=5)

            for bits in bits_list:
                serie = itaes_qga_dict.get(bits)
                if serie is None:
                    continue
                ax_ov.plot(np.arange(len(serie)), serie,
                           color=cores_bits[bits], linestyle=ls_bits[bits],
                           linewidth=2, label=f'QGA {bits}bits', alpha=0.9)

            # PT-BR: Linhas de threshold (relativas ao GA, para referência comum)
            # EN-US: Threshold lines (relative to GA, providing a common reference)
            for thr, cor_thr in cores_threshold.items():
                gen_thr = geracao_convergencia(itae_ga_serie, best_ga, thr)
                ax_ov.axvline(gen_thr, color=cor_thr, linestyle=':', alpha=0.5,
                              linewidth=1.2, label=f'GA {thr:.1f}%-conv gen {gen_thr}')

            ax_ov.set_title(f'Controlador {tipo_ctrl}', fontweight='bold', fontsize=13)
            ax_ov.set_xlabel('Geração')
            ax_ov.set_ylabel('Melhor ITAE')
            ax_ov.set_yscale('log')
            ax_ov.legend(fontsize=9, loc='upper right')
            ax_ov.grid(True, alpha=0.3)

        plt.tight_layout(rect=[0, 0, 1, 0.94])
        plt.savefig(FIGURES_DIR / 'convergencia_overlay_ga_vs_qga_bits.pdf', dpi=300, bbox_inches='tight')
        plt.show()
        print("Gráfico overlay salvo em 'convergencia_overlay_ga_vs_qga_bits.pdf'")

In [ ]:
for tipo_ctrl, itae_ga_serie, best_ga, itaes_qga_dict in [
            ('FT1',  itae_ga_t1,  best_ga_t1,  itaes_qga_t1),
            ('IT2-FLC', itae_ga_t2i, best_ga_t2i, itaes_qga_t2i),
        ]:
            import matplotlib.ticker as ticker
            # PT-BR: Garantindo que o numpy está importado
            # EN-US: Ensures that NumPy is imported
            import numpy as np

            # PT-BR: 1. Tamanho A4 ideal
            # EN-US: 1. Suitable A4 figure size
            fig, ax = plt.subplots(figsize=(9, 4.5)) 
            
            # PT-BR: FATOR DE ESCALA PARA LIMPAR O EIXO Y
            # EN-US: SCALING FACTOR FOR A CLEANER Y-AXIS
            fator = 1e4
            
            # PT-BR: 2. Título enxuto e direto ao ponto
            # EN-US: 2. Concise, focused title
            ax.set_title(
                f'Análise de Convergência: GA vs. QGA (Controlador {tipo_ctrl})',
                fontsize=12, fontweight='bold', pad=10
            )

            # PT-BR: ── GA Clássico ──────────────────────────────────────────────────
            # EN-US: ── Standard GA ───────────────────────────────────────────────
            gen_ga_1pct = geracao_convergencia(itae_ga_serie, best_ga, 1.0)

            ax.plot(
                # PT-BR: <-- Convertido para numpy array
                # EN-US: <-- Converted to a NumPy array
                np.arange(len(itae_ga_serie)), np.array(itae_ga_serie) * fator,
                color='steelblue', linewidth=2.5,
                label='GA', zorder=5 
            )
            ax.plot(
                # PT-BR: Aqui é só um número, então funciona direto
                # EN-US: This value is a scalar, so the direct operation is valid
                gen_ga_1pct, itae_ga_serie[gen_ga_1pct] * fator,
                marker='o', markersize=10,
                color='steelblue', markeredgecolor='white',
                markeredgewidth=1.5, zorder=10, linestyle='None'
            )

            # PT-BR: ── QGA por bits ─────────────────────────────────────────────────
            # EN-US: ── QGA by bit resolution ─────────────────────────────────────
            todas_series    = {'GA': (itae_ga_serie, 'steelblue', gen_ga_1pct)}
            info_conv_table = {'GA': gen_ga_1pct}

            for bits in bits_list:
                serie = itaes_qga_dict.get(bits)
                if serie is None:
                    continue

                best_qga     = min(serie)
                gen_qga_1pct = geracao_convergencia(serie, best_qga, 1.0)
                cor          = cores_bits[bits]

                ax.plot(
                    # PT-BR: <-- Convertido para numpy array
                    # EN-US: <-- Converted to a NumPy array
                    np.arange(len(serie)), np.array(serie) * fator,
                    color=cor, linestyle='-',
                    linewidth=2, label=f'QGA {bits} bits',
                    alpha=0.9, zorder=4
                )
                ax.plot(
                    gen_qga_1pct, serie[gen_qga_1pct] * fator, 
                    marker='o', markersize=10,
                    color=cor, markeredgecolor='white',
                    markeredgewidth=1.5, zorder=10, linestyle='None'
                )

                todas_series[f'QGA {bits}b']    = (serie, cor, gen_qga_1pct)
                info_conv_table[f'QGA {bits}b'] = gen_qga_1pct

            # PT-BR: ── Anotação discreta ──
            # EN-US: ── Subtle annotation ──
            for nome, (serie, cor, gen_conv) in todas_series.items():
                ax.annotate(
                    f'gen {gen_conv}',
                    xy=(gen_conv, serie[gen_conv] * fator), 
                    fontsize=9, color=cor, fontweight='bold',
                    xytext=(6, 6), textcoords='offset points',
                    va='bottom', ha='left',
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.95),
                    zorder=20
                )

            # PT-BR: 3. Nomes dos eixos limpos
            # EN-US: 3. Clear axis labels
            ax.set_xlabel('Geração', fontsize=11)
            ax.set_ylabel(r'Melhor ITAE ($\times 10^{-4}$)', fontsize=11) 
            ax.set_yscale('log')

            # PT-BR: 4. Controle absoluto do Eixo X
            # EN-US: 4. Explicit control of the x-axis
            ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
            ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
            
            # PT-BR: Grade e Eixo Y (Log)
            # EN-US: Grid and y-axis (logarithmic)
            ax.yaxis.set_minor_locator(ticker.LogLocator(subs='all', numticks=20))
            ax.grid(True, which='major', alpha=0.45, linestyle='-',  linewidth=0.8)
            ax.grid(True, which='minor', alpha=0.20, linestyle='--', linewidth=0.5)

            ax.tick_params(axis='x', which='minor', length=4)
            ax.tick_params(axis='both', which='major', labelsize=10)

            # PT-BR: 5. Formatação manual absoluta para forçar números decimais puros (sem 10^x)
            # EN-US: 5. Explicit manual formatting to force plain decimal labels (without 10^x)
            formatter = ticker.FormatStrFormatter('%.2f')
            ax.yaxis.set_major_formatter(formatter)
            ax.yaxis.set_minor_formatter(formatter)
            
            # PT-BR: 6. Legenda
            # EN-US: 6. Legend
            ax.legend(
                title='\u25cf Convergência (1%)',
                title_fontsize=10,
                fontsize=10, 
                loc='upper right',
                framealpha=0.95, 
                ncol=1
            )

            plt.tight_layout() 
            fname = f'convergencia_overlay_1pct_{tipo_ctrl}_ga_vs_qga.pdf'
            plt.savefig(FIGURES_DIR / fname, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"Gráfico {tipo_ctrl} salvo em '{fname}'")
            print(f"  Gerações de convergência (1%): {info_conv_table}")

In [ ]:
# PT-BR: Tabelas de convergência e estimativas por rateio
# EN-US: Convergence tables and proportional-allocation estimates

if not DATA_LOADED_SUCCESSFULLY:
    print("ERRO: Dados não carregados. Execute o bloco de carregamento primeiro.")
else:
    # -----------------------------------------------------------------------
    # PT-BR: Funções auxiliares
    # EN-US: Helper functions
    # -----------------------------------------------------------------------
    def extrair_itae_ga(historico):
        """PT-BR:
        Extrai ITAE por geração do histórico do GA Standard (lista de floats).

        EN-US:
        Extracts ITAE by generation from the Standard GA history (a list of floats).
        """
        return [float(v) for v in historico]

    def extrair_itae_qga(historico):
        """PT-BR:
        Extrai ITAE por geração do histórico do QGA (lista de dicts ou floats).

        EN-US:
        Extracts ITAE by generation from the QGA history (a list of dictionaries or floats).
        """
        itae_list = []
        for h in historico:
            if isinstance(h, dict):
                apt = h.get('melhor_aptidao_global', h.get('melhor_aptidao_geracao', 1e-9))
                itae = (1.0 / apt - 1e-9) if apt > 1e-9 else 1e9
            else:
                itae = float(h)
            itae_list.append(itae)
        return itae_list

    def geracao_convergencia(itae_serie, melhor_final, threshold_pct=1.0):
        """PT-BR:
        Retorna a primeira geração g onde ITAE(g) <= ITAE_melhor * (1 + L/100).
        Eq. (eq:itae_alvo) da subseção de critérios de convergência.

        EN-US:
        Returns the first generation g for which ITAE(g) <= ITAE_best * (1 + L/100).
        See Eq. (eq:itae_alvo) in the convergence-criteria subsection.
        """
        alvo = melhor_final * (1 + threshold_pct / 100.0)
        for i, v in enumerate(itae_serie):
            if v <= alvo:
                return i
        return len(itae_serie) - 1

    def tempo_convergencia(t_total, gen_conv, g_max):
        """PT-BR:
        Estima t_conv = t_total * (g_conv / G_max).
        Eq. (eq:tempo_convergencia) da subseção de critérios de convergência.

        EN-US:
        Estimates t_conv = t_total * (g_conv / G_max).
        See Eq. (eq:tempo_convergencia) in the convergence-criteria subsection.
        """
        return t_total * (gen_conv / max(g_max, 1))

    # -----------------------------------------------------------------------
    # PT-BR: GA Standard - histórico e tempos (referência de bits[0])
    # EN-US: Standard GA — history and run times (reference from bits[0])
    # -----------------------------------------------------------------------
    historico_ga_t1  = ga_historico_ga_t1  if 'ga_historico_ga_t1'  in globals() else None
    historico_ga_t2i = ga_historico_ga_t2i if 'ga_historico_ga_t2i' in globals() else None

    _t_ga_t1  = ga_tempo_ga_t1_s  if 'ga_tempo_ga_t1_s'  in globals() else 0.0
    _t_ga_t2i = ga_tempo_ga_t2i_s if 'ga_tempo_ga_t2i_s' in globals() else 0.0

    if historico_ga_t1 is None or historico_ga_t2i is None:
        print("AVISO: Histórico do GA Standard não encontrado.")
        print([k for k in globals() if k.startswith('ga_') or k.startswith('qga_')])
    else:
        itae_ga_t1  = extrair_itae_ga(historico_ga_t1[1:])
        itae_ga_t2i = extrair_itae_ga(historico_ga_t2i[1:])
        best_ga_t1  = min(itae_ga_t1)
        best_ga_t2i = min(itae_ga_t2i)

        # -----------------------------------------------------------------------
        # PT-BR: QGA - histórico e tempos para cada resolução de bits
        # EN-US: QGA — history and run times for each bit resolution
        # -----------------------------------------------------------------------
        cores_bits = {10: 'crimson', 16: 'darkorange', 24: 'purple', 32: 'saddlebrown'}
        ls_bits    = {10: '-',       16: '-.',          24: (0,(3,1,1,1)), 32: ':'}

        historicos_qga_t1  = {}
        historicos_qga_t2i = {}
        tempos_qga_t1      = {}
        tempos_qga_t2i     = {}
        itaes_qga_t1       = {}
        itaes_qga_t2i      = {}
        bests_qga_t1       = {}
        bests_qga_t2i      = {}

        for bits in bits_list:
            dados_bits = all_dados.get(bits) or {}
            hist_t1  = dados_bits.get('qga_historico_qga_t1')
            hist_t2i = dados_bits.get('qga_historico_qga_t2i')
            t_t1     = dados_bits.get('qga_tempo_qga_t1_s',  0.0)
            t_t2i    = dados_bits.get('qga_tempo_qga_t2i_s', 0.0)

            historicos_qga_t1[bits]  = hist_t1
            historicos_qga_t2i[bits] = hist_t2i
            tempos_qga_t1[bits]      = t_t1
            tempos_qga_t2i[bits]     = t_t2i

            if hist_t1 is not None:
                itaes_qga_t1[bits] = extrair_itae_qga(hist_t1)
                bests_qga_t1[bits] = min(itaes_qga_t1[bits])
            if hist_t2i is not None:
                itaes_qga_t2i[bits] = extrair_itae_qga(hist_t2i)
                bests_qga_t2i[bits] = min(itaes_qga_t2i[bits])

        # -----------------------------------------------------------------------
        # PT-BR: Limiares de tolerância: L = 1%, 2%, 5%  (subseção eq:itae_alvo)
        # EN-US: Tolerance thresholds: L = 1%, 2%, 5% (subsection eq:itae_alvo)
        # -----------------------------------------------------------------------
        thresholds = [5.0, 2.0, 1.0]

        # -----------------------------------------------------------------------
        # PT-BR: TABELA RESUMO - Métricas gerais (tempo total e gerações)
        # EN-US: SUMMARY TABLE — overall metrics (total time and generations)
        # -----------------------------------------------------------------------
        print("\n" + "="*90)
        print("ANÁLISE DE CONVERGÊNCIA ANTECIPADA: GA Standard vs. QGA (multi-bits)")
        print("="*90)

        cab = f"{'Métrica':<32} {'GA Standard':>14}"
        for bits in bits_list:
            cab += f"  {'QGA '+str(bits)+'b':>12}"
        print(cab)
        print("-"*90)

        # PT-BR: Melhor ITAE final - T1
        # EN-US: Final best ITAE — T1
        linha = f"{'Melhor ITAE T1':<32} {best_ga_t1:>14.4e}"
        for bits in bits_list:
            v = bests_qga_t1.get(bits)
            linha += f"  {(f'{v:.4e}' if v is not None else 'N/A'):>12}"
        print(linha)

        # PT-BR: Melhor ITAE final - T2I
        # EN-US: Final best ITAE — T2I
        linha = f"{'Melhor ITAE T2I':<32} {best_ga_t2i:>14.4e}"
        for bits in bits_list:
            v = bests_qga_t2i.get(bits)
            linha += f"  {(f'{v:.4e}' if v is not None else 'N/A'):>12}"
        print(linha)

        # PT-BR: Total de gerações - T1
        # EN-US: Total generations — T1
        linha = f"{'Gerações totais T1 (G_max)':<32} {len(itae_ga_t1):>14}"
        for bits in bits_list:
            s = itaes_qga_t1.get(bits)
            linha += f"  {(str(len(s)) if s is not None else 'N/A'):>12}"
        print(linha)

        # PT-BR: Total de gerações - T2I
        # EN-US: Total generations — T2I
        linha = f"{'Gerações totais T2I (G_max)':<32} {len(itae_ga_t2i):>14}"
        for bits in bits_list:
            s = itaes_qga_t2i.get(bits)
            linha += f"  {(str(len(s)) if s is not None else 'N/A'):>12}"
        print(linha)

        # PT-BR: Tempo total (wall-clock) - T1
        # EN-US: Total wall-clock time — T1
        linha = f"{'t_total T1 (min)':<32} {_t_ga_t1/60:>14.1f}"
        for bits in bits_list:
            t = tempos_qga_t1.get(bits, 0.0)
            linha += f"  {t/60:>12.1f}"
        print(linha)

        # PT-BR: Tempo total (wall-clock) - T2I
        # EN-US: Total wall-clock time — T2I
        linha = f"{'t_total T2I (min)':<32} {_t_ga_t2i/60:>14.1f}"
        for bits in bits_list:
            t = tempos_qga_t2i.get(bits, 0.0)
            linha += f"  {t/60:>12.1f}"
        print(linha)

        # -----------------------------------------------------------------------
        # PT-BR: CONVERGÊNCIA POR LIMIAR - T1 e T2I
        # EN-US: THRESHOLD-BASED CONVERGENCE — T1 and T2I
        # PT-BR: L = 5%, 2%, 1%  conforme eq:itae_alvo / eq:tempo_convergencia
        # EN-US: L = 5%, 2%, 1% according to eq:itae_alvo / eq:tempo_convergencia
        # -----------------------------------------------------------------------
        for tipo_ctrl, itae_ga_serie, best_ga, itaes_qga_dict, tempos_qga_dict, t_ga_total in [
            ('T1',  itae_ga_t1,  best_ga_t1,  itaes_qga_t1,  tempos_qga_t1,  _t_ga_t1),
            ('T2I', itae_ga_t2i, best_ga_t2i, itaes_qga_t2i, tempos_qga_t2i, _t_ga_t2i),
        ]:
            g_max_ga = len(itae_ga_serie)
            print(f"\n{'-'*90}")
            print(f"  Controlador {tipo_ctrl} - Convergência por limiar L")
            print(f"  t_total GA = {t_ga_total/60:.1f} min | G_max GA = {g_max_ga} gerações")
            print(f"{'-'*90}")

            for L in thresholds:
                g_conv_ga = geracao_convergencia(itae_ga_serie, best_ga, L)
                t_conv_ga = tempo_convergencia(t_ga_total, g_conv_ga, g_max_ga)

                print(f"\n   L = {L:.0f}%  →  ITAE_alvo = ITAE_melhor × (1 + {L:.0f}/100)")
                print(f"    {'Algoritmo':<22} {'g_conv':>8} {'g_conv/G_max':>14} {'t_conv (min)':>14} {'vs GA':>12}")
                print(f"    {'-'*72}")
                print(f"    {'GA Standard':<22} {g_conv_ga:>8} "
                      f"{g_conv_ga/g_max_ga:>14.3f} {t_conv_ga/60:>14.1f} {'-':>12}")

                for bits in bits_list:
                    serie = itaes_qga_dict.get(bits)
                    t_qga = tempos_qga_dict.get(bits, 0.0)
                    if serie is None:
                        print(f"    {'QGA '+str(bits)+'bits':<22} {'N/A':>8} {'N/A':>14} {'N/A':>14} {'N/A':>12}")
                        continue
                    best_qga   = min(serie)
                    g_max_qga  = len(serie)
                    g_conv_qga = geracao_convergencia(serie, best_qga, L)
                    t_conv_qga = tempo_convergencia(t_qga, g_conv_qga, g_max_qga)
                    ganho_pct  = ((t_conv_ga - t_conv_qga) / max(t_conv_ga, 1e-9)) * 100
                    sinal      = f"({ganho_pct:+.1f}%)"
                    print(f"    {'QGA '+str(bits)+'bits':<22} {g_conv_qga:>8} "
                          f"{g_conv_qga/g_max_qga:>14.3f} {t_conv_qga/60:>14.1f} {sinal:>12}")

        # -----------------------------------------------------------------------
        # PT-BR: REFERÊNCIA CRUZADA: geração em que QGA atingiu o melhor ITAE do GA
        # EN-US: CROSS-REFERENCE: generation in which QGA reached the best GA ITAE
        # PT-BR: (L = 0%, limiar exato)
        # EN-US: (L = 0%, exact threshold)
        # -----------------------------------------------------------------------
        print(f"\n{'-'*90}")
        print("  Referência cruzada: geração em que QGA igualou o melhor ITAE do GA Standard (L=0%)")
        print(f"{'-'*90}")
        for tipo_ctrl, itae_ga_serie, best_ga, itaes_qga_dict, tempos_qga_dict, t_ga_total in [
            ('T1',  itae_ga_t1,  best_ga_t1,  itaes_qga_t1,  tempos_qga_t1,  _t_ga_t1),
            ('T2I', itae_ga_t2i, best_ga_t2i, itaes_qga_t2i, tempos_qga_t2i, _t_ga_t2i),
        ]:
            print(f"\n  [{tipo_ctrl}] Melhor GA = {best_ga:.4e}  |  t_total GA = {t_ga_total/60:.1f} min")
            for bits in bits_list:
                serie = itaes_qga_dict.get(bits)
                t_qga = tempos_qga_dict.get(bits, 0.0)
                if serie is None:
                    print(f"    QGA {bits}bits - dados não disponíveis")
                    continue
                g_vs   = geracao_convergencia(serie, best_ga, threshold_pct=0.0)
                t_vs   = tempo_convergencia(t_qga, g_vs, len(serie))
                ganho  = ((t_ga_total - t_vs) / max(t_ga_total, 1e-9)) * 100
                print(f"    QGA {bits}bits: g_conv={g_vs}  t_conv={t_vs/60:.1f} min  "
                      f"ITAE={serie[g_vs]:.4e}  ganho={ganho:+.1f}% vs t_total GA")